# MathScholar — Stage C: table-focused continuation fine-tune

Continues fine-tuning from the saved `curve_n122` adapter (Step 28's best checkpoint),
table-focused, per the direct request after Step 28's curve showed strong formula/prose
performance but hallucination on dense numeric tables (`as_p0509`, `as_p0351`).

Training data: 9 real NIST table pairs (embedded below) + 26 already-verified table-dense
A&S train pages (materialized via `ANNOT=1 get_data.sh`, same as every other GPU step).

**Catastrophic-forgetting guard**: writes to a NEW adapter directory
(`data/models/ocr_lora/table_ft/`), never overwrites `curve_n122`; low LR; early-stops on
the full 20-page A&S validation set, not a table-only subset; prints a per-page
before/after comparison so any regression is visible, not averaged away.

Generated by `scripts/build_kaggle_notebook_stage_c.py` from
`scripts/run_finetune_tables.py` — regenerate from there, don't hand-edit this file.


In [ ]:
REPO_URL = "https://github.com/anuronmaitro/doc-agent-1.git"
BRANCH = "main"
ADAPTER_DATASET_SLUG = "mathscholar-curve-n122-adapter"


## 1. Clone the repo and install pinned dependencies

In [ ]:
import os, subprocess
if not os.path.exists("/kaggle/working/repo"):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, "/kaggle/working/repo"], check=True)
%cd /kaggle/working/repo
!pip install -q --no-cache-dir -r requirements.lock
import pkg_resources  # noqa: F401
print("pkg_resources OK")


## 2. Mount the starting checkpoint (curve_n122)

Attach the `mathscholar-curve-n122-adapter` dataset to this kernel (Add Input) before
running. Version 1 crashed here (`FileNotFoundError:
/kaggle/input/mathscholar-curve-n122-adapter`); version 2's fuzzy name-match on
`/kaggle/input/*` crashed too, because on this (papermill/batch-executed) kernel the
dataset is nested one level deeper than the interactive-notebook convention --
`/kaggle/input/` only contains a single `datasets` entry, with the real per-dataset folder
somewhere underneath it. Rather than guess a third exact path shape, this walks the whole
`/kaggle/input` tree and finds whichever directory actually contains `adapter_config.json`
-- the file that uniquely identifies the PEFT adapter's own contents -- so it's independent
of naming/nesting quirks entirely.

In [ ]:
import shutil, os

print("Available /kaggle/input/ entries:", os.listdir("/kaggle/input"))
adapter_dir = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "adapter_config.json" in files:
        adapter_dir = root
        break
if adapter_dir is None:
    raise FileNotFoundError(
        "No directory under /kaggle/input contains adapter_config.json -- attach the "
        "'mathscholar-curve-n122-adapter' dataset to this kernel (Add Input) before "
        "running. Full tree: "
        f"{[(r, files) for r, _, files in os.walk('/kaggle/input')]}"
    )
src = adapter_dir
print(f"using {src}")
dst = "data/models/ocr_lora/curve_n122"
os.makedirs(dst, exist_ok=True)
shutil.copytree(src, dst, dirs_exist_ok=True)
print("copied", os.listdir(dst))


## 3. Materialize the A&S page images (val + the 26 table-dense train pages)

Same `ANNOT=1 bash scripts/get_data.sh` every GPU step uses -- covers all 181 annotation
pages (val + train + test) in one pass, so this also materializes the 26 table-dense
train pages `run_finetune_tables.py` reuses, not just val.

In [ ]:
!ANNOT=1 bash scripts/get_data.sh


## 4. Write the NIST table pairs (9 pairs, embedded -- small enough to inline, see
this notebook's own generator docstring for why it isn't a Dataset like the adapter).

In [ ]:
import base64, os
os.makedirs("data/annot/nist_tables/images", exist_ok=True)
open("data/annot/nist_tables/images/nisttbl_p0097_3-5-10.png", "wb").write(base64.b64decode("..."))
open("data/annot/nist_tables/images/nisttbl_p0097_3-5-11.png", "wb").write(base64.b64decode("..."))
open("data/annot/nist_tables/images/nisttbl_p0097_3-5-6.png", "wb").write(base64.b64decode("..."))
open("data/annot/nist_tables/images/nisttbl_p0097_3-5-7.png", "wb").write(base64.b64decode("..."))
open("data/annot/nist_tables/images/nisttbl_p0098_3-5-14.png", "wb").write(base64.b64decode("..."))
open("data/annot/nist_tables/images/nisttbl_p0098_3-5-15.png", "wb").write(base64.b64decode("..."))
open("data/annot/nist_tables/images/nisttbl_p0099_3-5-18.png", "wb").write(base64.b64decode("..."))
open("data/annot/nist_tables/images/nisttbl_p0110_3-9-1.png", "wb").write(base64.b64decode("..."))
open("data/annot/nist_tables/images/nisttbl_p0655_27-2-1.png", "wb").write(base64.b64decode("..."))
print(f"wrote {len(os.listdir('data/annot/nist_tables/images'))} table images")

In [ ]:
%%writefile data/annot/nist_tables/pairs.jsonl
{"pair_id": "nisttbl_p0097_3-5-6", "pdf_page": 97, "table_id": "3.5.6", "text": "Table 3.5.6: Nodes and weights for the 5-point Gauss–Laguerre formula.\nxk  wk\n0.26356 03197 18141  0.52175 56105 82809\n0.14134 03059 10652×101  0.39866 68110 83176\n0.35964 25771 04072×101  0.75942 44968 17076×10−1\n0.70858 10005 85884×101  0.36117 58679 92205×10−2\n0.12640 80084 42758×102  0.23369 97238 57762×10−4", "image": "data/annot/nist_tables/images/nisttbl_p0097_3-5-6.png"}
{"pair_id": "nisttbl_p0097_3-5-7", "pdf_page": 97, "table_id": "3.5.7", "text": "Table 3.5.7: Nodes and weights for the 10-point Gauss–Laguerre formula.\nxk  wk\n0.13779 34705 40492 431  0.30844 11157 65020 141\n0.72945 45495 03170 498  0.40111 99291 55273 552\n0.18083 42901 74031 605×101  0.21806 82876 11809 422\n0.34014 33697 85489 951×101  0.62087 45609 86777 475×10−1\n0.55524 96140 06380 363×101  0.95015 16975 18110 055×10−2\n0.83301 52746 76449 670×101  0.75300 83885 87538 775×10−3\n0.11843 78583 79000 656×102  0.28259 23349 59956 557×10−4\n0.16279 25783 13781 021×102  0.42493 13984 96268 637×10−6\n0.21996 58581 19807 620×102  0.18395 64823 97963 078×10−8\n0.29920 69701 22738 916×102  0.99118 27219 60900 856×10−12", "image": "data/annot/nist_tables/images/nisttbl_p0097_3-5-7.png"}
{"pair_id": "nisttbl_p0097_3-5-10", "pdf_page": 97, "table_id": "3.5.10", "text": "Table 3.5.10: Nodes and weights for the 5-point Gauss–Hermite formula.\n±xk  wk\n0.00000 00000 00000  0.94530 87204 82942\n0.95857 24646 13819  0.39361 93231 52241\n0.20201 82870 45609×101  0.19953 24205 90459×10−1", "image": "data/annot/nist_tables/images/nisttbl_p0097_3-5-10.png"}
{"pair_id": "nisttbl_p0097_3-5-11", "pdf_page": 97, "table_id": "3.5.11", "text": "Table 3.5.11: Nodes and weights for the 10-point Gauss–Hermite formula.\n±xk  wk\n0.34290 13272 23704 609  0.61086 26337 35325 799\n0.10366 10829 78951 365×101  0.24013 86110 82314 686\n0.17566 83649 29988 177×101  0.33874 39445 54810 631×10−1\n0.25327 31674 23278 980×101  0.13436 45746 78123 269×10−2\n0.34361 59118 83773 760×101  0.76404 32855 23262 063×10−5", "image": "data/annot/nist_tables/images/nisttbl_p0097_3-5-11.png"}
{"pair_id": "nisttbl_p0098_3-5-14", "pdf_page": 98, "table_id": "3.5.14", "text": "Table 3.5.14: Nodes and weights for the 5-point Gauss formula for the logarithmic weight function.\nxk  wk\n0.29134 47215 19721×10−1  0.29789 34717 82894\n0.17397 72133 20898  0.34977 62265 13224\n0.41170 25202 84902  0.23448 82900 44052\n0.67731 41745 82820  0.98930 45951 66331×10−1\n0.89477 13610 31008  0.18911 55214 31958×10−1", "image": "data/annot/nist_tables/images/nisttbl_p0098_3-5-14.png"}
{"pair_id": "nisttbl_p0098_3-5-15", "pdf_page": 98, "table_id": "3.5.15", "text": "Table 3.5.15: Nodes and weights for the 10-point Gauss formula for the logarithmic weight function.\nxk  wk\n0.90426 30962 19965 064×10−2  0.12095 51319 54570 515\n0.53971 26622 25006 295×10−1  0.18636 35425 64071 870\n0.13531 18246 39250 775  0.19566 08732 77759 983\n0.24705 24162 87159 824  0.17357 71421 82906 921\n0.38021 25396 09332 334  0.13569 56729 95484 202\n0.52379 23179 71843 201  0.93646 75853 81105 260×10−1\n0.66577 52055 16424 597  0.55787 72735 14158 741×10−1\n0.79419 04160 11966 217  0.27159 81089 92333 311×10−1\n0.89816 10912 19003 538  0.95151 82602 84851 500×10−2\n0.96884 79887 18633 539  0.16381 57633 59826 325×10−2", "image": "data/annot/nist_tables/images/nisttbl_p0098_3-5-15.png"}
{"pair_id": "nisttbl_p0099_3-5-18", "pdf_page": 99, "table_id": "3.5.18", "text": "Table 3.5.18: Nodes and weights for the 5-point complex Gauss quadrature formula with s = 1.\nζk  wk\n3.65569 4325+6.54373 6899i  3.83966 1630−0.27357 03863i\n3.65569 4325−6.54373 6899i  3.83966 1630+0.27357 03863i\n5.70095 3299+3.21026 5600i  −25.07945 221 +2.18725 2294i\n5.70095 3299−3.21026 5600i  −25.07945 221 −2.18725 2294i\n6.28670 4752+0.00000 0000i  43.47958 116 +0.00000 0000i", "image": "data/annot/nist_tables/images/nisttbl_p0099_3-5-18.png"}
{"pair_id": "nisttbl_p0110_3-9-1", "pdf_page": 110, "table_id": "3.9.1", "text": "Table 3.9.1: Shanks’ transformation for sn = Pn\ntn,2  tn,4  tn,6  tn,8  tn,10\n0.80000 00000 00  0.82182 62806 24  0.82244 84501 47  0.82246 64909 60  0.82246 70175 41\n0.82692 30769 23  0.82259 02017 65  0.82247 05346 57  0.82246 71342 06  0.82246 70363 45\n0.82111 11111 11  0.82243 44785 14  0.82246 61821 45  0.82246 70102 48  0.82246 70327 79\n0.82300 13550 14  0.82247 78118 35  0.82246 72851 83  0.82246 70397 56  0.82246 70335 90\n0.82221 76684 88  0.82246 28314 41  0.82246 69467 93  0.82246 70314 36  0.82246 70333 75\n0.82259 80392 16  0.82246 88857 22  0.82246 70670 21  0.82246 70341 24  0.82246 70334 40\n0.82239 19390 77  0.82246 61352 37  0.82246 70190 76  0.82246 70331 54  0.82246 70334 18\n0.82251 30483 23  0.82246 75033 13  0.82246 70400 56  0.82246 70335 37  0.82246 70334 26\n0.82243 73137 33  0.82246 67719 32  0.82246 70301 49  0.82246 70333 73  0.82246 70334 23\n0.82248 70624 89  0.82246 71865 91  0.82246 70351 34  0.82246 70334 48  0.82246 70334 24\n0.82245 30535 15  0.82246 69397 57  0.82246 70324 88  0.82246 70334 12  0.82246 70334 24", "image": "data/annot/nist_tables/images/nisttbl_p0110_3-9-1.png"}
{"pair_id": "nisttbl_p0655_27-2-1", "pdf_page": 655, "table_id": "27.2.1", "text": "Table 27.2.1: Primes.\npn+20  pn+30  pn+40  pn+50  pn+60  pn+70  pn+80  pn+90\n73  127  179  233  283  353  419  467\n79  131  181  239  293  359  421  479\n83  137  191  241  307  367  431  487\n89  139  193  251  311  373  433  491\n97  149  197  257  313  379  439  499\n101  151  199  263  317  383  443  503\n103  157  211  269  331  389  449  509\n107  163  223  271  337  397  457  521\n109  167  227  277  347  401  461  523\n113  173  229  281  349  409  463  541", "image": "data/annot/nist_tables/images/nisttbl_p0655_27-2-1.png"}


In [ ]:
%%writefile data/annot/nist_tables/summary.json
{
  "source_pdf": "data/raw/NIST_Handbook_Mathematical_Functions.pdf",
  "pdf_pages_scanned": 967,
  "pages_with_tables": 5,
  "table_pairs_written": 9,
  "out_dir": "data/annot/nist_tables",
  "render_dpi": 300,
  "elapsed_s": 4.4
}


## 5. Write the code (5 files -- run_finetune.py is imported by
run_finetune_tables.py for its shared eval/adapter helpers).

In [ ]:
import os
os.makedirs("src/doc_agent/training", exist_ok=True)
os.makedirs("src/doc_agent/vision", exist_ok=True)


In [ ]:
%%writefile src/doc_agent/training/datamodule.py
"""Training — Lightning datamodule.

Two data stages, matching plan.md Step 28's "two-stage fine-tune" (Stage A -> Stage B, not
one mixed dataset — see `train.py`, which runs `Trainer.fit()` twice against two separate
`DocDataModule` instances built from this file, continuing the SAME `LitComponent` weights
across both calls):

- **Stage A ("nist")** — Step 25's 695 NIST formula crops, degraded ON THE FLY per sample
  (see `_NistStageADataset`) using the exact same ops `scripts/degrade_nist_pairs.py` uses
  for its committed report figures (`doc_agent.training.degrade`, extracted from that
  script at this step so the two never drift apart). No pre-materialized
  `data/interim/nist_degraded/` directory is required or read — training never depends on
  that directory existing, only on Step 25's `data/annot/nist/pairs.jsonl` +
  `images/*.png`, both committed. No validation split (see plan.md Step 27's DECISION:
  Stage A is off-distribution volume/warmup, not the model-selection signal).
- **Stage B ("as")** — the 122 A&S hand-annotated train pages + 20 val pages
  (`data/annot/train|val/*.json` + their sibling `.png`). The chapter-disjoint split is
  already encoded by which directory a page's JSON lives in (Steps 18b-24's
  `doc_agent.data.validate.ANNOT_SETS`); `setup()` re-asserts each loaded page's chapter
  against that lock rather than re-deriving the split, so a corrupted/misplaced file fails
  loudly here instead of silently leaking chapters between train and val.
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Literal

import lightning as L
import numpy as np
from PIL import Image
from torch.utils.data import DataLoader, Dataset

from ..contracts import *  # noqa
from ..logging_conf import get_logger
from .degrade import degrade_one

logger = get_logger(__name__)


def _load_yaml(path: str) -> dict[str, Any]:
    import yaml

    with open(path, encoding="utf-8") as fh:
        return yaml.safe_load(fh)


def _load_jsonl(path: str) -> list[dict[str, Any]]:
    records = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


class _NistStageADataset(Dataset):
    """Stage A: one NIST formula crop per item, degraded on-the-fly (never from a
    pre-materialized directory) so training has no dependency on
    `scripts/degrade_nist_pairs.py` having been run first."""

    def __init__(self, pairs_path: str, degradation_cfg: dict[str, Any], seed: int) -> None:
        self._pairs = _load_jsonl(pairs_path)
        if not self._pairs:
            raise ValueError(f"_NistStageADataset: {pairs_path} is empty")
        self._deg_cfg = degradation_cfg
        self._seed = seed
        # Bumped by `SeedByEpochCallback` before each epoch (default 0, i.e. unchanged
        # behavior for a 1-epoch run). See that callback's docstring for why this exists:
        # without it, every epoch beyond the first would replay the IDENTICAL degraded
        # image per pair rather than a new augmented variant.
        self.epoch = 0

    def __len__(self) -> int:
        return len(self._pairs)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        rec = self._pairs[idx]
        src = Image.open(rec["image"]).convert("L")
        arr = np.asarray(src, dtype=np.float32)
        # Per-sample-seeded RNG (config seed + pair index + epoch * dataset size) -- any
        # given (index, epoch) pair's degraded image is reproducible across runs/workers
        # without needing to persist it. The epoch term was added at Step 28 (found running
        # the real multi-epoch Stage A on Kaggle, see SeedByEpochCallback): without it this
        # was `seed + idx` alone, identical every epoch regardless of how many epochs ran --
        # fine for the 1-epoch default this shipped with, silently wrong for >1.
        rng = np.random.default_rng(self._seed + idx + self.epoch * len(self._pairs))
        degraded = degrade_one(arr, self._deg_cfg, rng)
        image = Image.fromarray(degraded).convert("RGB")
        return {"image": image, "text": rec["text"]}


class SeedByEpochCallback(L.Callback):
    """Advances `_NistStageADataset.epoch` before each training epoch, so a multi-epoch
    Stage A run degrades each pair differently per epoch instead of replaying the same
    image (see that dataset's own comment for the bug this fixes -- found running the real
    Step 28 Kaggle GPU job with `stage_a.max_epochs > 1`). No-op for a 1-epoch run (Step
    27's own smoke test and default config), since `on_train_epoch_start` only ever sets
    `epoch = 0` there -- safe to always attach, not something that needs conditioning on
    `max_epochs`."""

    def __init__(self, dataset: "_NistStageADataset") -> None:
        self._dataset = dataset

    def on_train_epoch_start(self, trainer: Any, pl_module: Any) -> None:  # noqa: ARG002
        self._dataset.epoch = trainer.current_epoch


class _ASStageBDataset(Dataset):
    """Stage B: one A&S annotated page per item -- the hand-corrected `text` (full page,
    real backslash-LaTeX) paired with that page's own 300dpi grayscale render."""

    def __init__(
        self,
        annot_dir: str,
        expected_chapters: frozenset[str] | None = None,
        max_pages: int | None = None,
    ) -> None:
        from ..data.validate import ANNOT_EXPECTED_COUNTS
        from ..ingest.loader import _chapter_of

        split = Path(annot_dir).name  # "train" or "val"
        json_paths = sorted(Path(annot_dir).glob("*.json"))
        if not json_paths:
            raise FileNotFoundError(
                f"_ASStageBDataset: no *.json under {annot_dir} -- run "
                "`ANNOT=1 bash scripts/get_data.sh` first if this is a fresh clone "
                "(images are gitignored; see .gitignore's data/annot/val|train comment)"
            )
        expected = ANNOT_EXPECTED_COUNTS.get(split)
        if max_pages is None and expected is not None and len(json_paths) != expected:
            raise ValueError(
                f"_ASStageBDataset: {annot_dir} has {len(json_paths)} pages, expected "
                f"{expected} (doc_agent.data.validate.ANNOT_EXPECTED_COUNTS[{split!r}]) -- "
                "a missing or extra file here silently leaks/shrinks the fine-tune's "
                "train/val split, so this is a hard error, not a warning"
            )
        if max_pages is not None:
            # Step 28's learning curve (25/50/105/122 train pages): a fixed sort order
            # (page_id, already the glob's sort key) makes each smaller curve point a
            # PREFIX of every larger one -- 25 pages ⊂ 50 ⊂ 105 ⊂ 122 -- rather than an
            # independently-resampled subset, so successive curve points differ only by
            # which pages were ADDED, which is what makes "did going from 105->122 help"
            # (plan.md Step 28 point 2) a meaningful comparison instead of confounded by
            # also swapping out which pages were included. Never applied to val (the
            # 20-page val set stays whole and identical across every curve point, which is
            # what makes the curve's y-axis comparable point to point).
            json_paths = json_paths[:max_pages]

        self._records: list[dict[str, Any]] = []
        for jp in json_paths:
            row = json.loads(jp.read_text(encoding="utf-8"))
            png_path = jp.with_suffix(".png")
            if not png_path.exists():
                raise FileNotFoundError(
                    f"_ASStageBDataset: {jp} has no sibling image {png_path} -- "
                    "run `ANNOT=1 bash scripts/get_data.sh` to materialize it "
                    "(gitignored, reproducible, byte-identical to data/pages/)"
                )
            if expected_chapters is not None:
                actual = _chapter_of(row["printed_page"])
                if actual not in expected_chapters:
                    raise ValueError(
                        f"_ASStageBDataset: LEAK — {row['page_id']} (chapter {actual}) is "
                        f"in {annot_dir} but that chapter is not one of {split}'s allowed "
                        f"chapters. See doc_agent.data.validate.ANNOT_SETS."
                    )
            if not row.get("text", "").strip():
                # A page whose annotation JSON exists but was never filled in (text=="")
                # is an in-progress annotation, not a training sample -- silently training
                # on an empty target would just teach the model to predict nothing.
                raise ValueError(
                    f"_ASStageBDataset: {jp} has empty text -- annotation incomplete, "
                    "not ready to train on"
                )
            self._records.append({"image_path": str(png_path), "text": row["text"]})

    def __len__(self) -> int:
        return len(self._records)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        rec = self._records[idx]
        image = Image.open(rec["image_path"]).convert("RGB")
        return {"image": image, "text": rec["text"]}


def make_collate_fn(processor: Any, max_target_length: int) -> Any:
    """Builds a collate_fn closing over a loaded NougatProcessor -- kept as a factory
    (not a bare module-level function) since the processor must be loaded once by
    `LitComponent` and shared with the datamodule, not reloaded per batch."""

    def collate(batch: list[dict[str, Any]]) -> dict[str, Any]:
        images = [b["image"] for b in batch]
        texts = [b["text"] for b in batch]
        pixel_values = processor(images, return_tensors="pt").pixel_values
        enc = processor.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_target_length,
        )
        labels = enc.input_ids.clone()
        labels[enc.attention_mask == 0] = -100  # ignore padding in the loss
        return {"pixel_values": pixel_values, "labels": labels}

    return collate


class DocDataModule(L.LightningDataModule):
    """`data_stage` selects Stage A ("nist") or Stage B ("as") -- see module docstring for
    why these are two separate DocDataModule instances rather than one mixed dataset.
    `collate_fn` is injected (not built internally) so it shares `LitComponent`'s already-
    loaded processor instead of this datamodule loading a second copy of it."""

    def __init__(
        self,
        cfg: dict[str, Any],
        data_stage: Literal["nist", "as"],
        collate_fn: Any,
    ) -> None:
        super().__init__()
        self.cfg = cfg
        self.data_stage = data_stage
        self.collate_fn = collate_fn
        self.train_dataset: Dataset | None = None
        self.val_dataset: Dataset | None = None

    def setup(self, stage: str | None = None) -> None:
        data_cfg = self.cfg["data"]
        if self.data_stage == "nist":
            degradation_cfg = _load_yaml(data_cfg["degradation_cfg"])
            self.train_dataset = _NistStageADataset(
                data_cfg["nist_pairs_path"], degradation_cfg, self.cfg["seed"]
            )
            self.val_dataset = None
            logger.info(f"DocDataModule[nist]: {len(self.train_dataset)} Stage A crops")
        elif self.data_stage == "as":
            from ..data.validate import BUILD_CHAPTERS, VAL_CHAPTERS

            # Step 28's learning curve (25/50/105/122 train pages): set once, here, via
            # cfg["data"]["stage_b_max_train_pages"] -- never applied to val, so the same
            # 20 pages are used at every curve point (see _ASStageBDataset's own comment
            # on why that's what keeps the curve's points comparable).
            max_train_pages = data_cfg.get("stage_b_max_train_pages")
            self.train_dataset = _ASStageBDataset(
                data_cfg["train_annot_dir"], BUILD_CHAPTERS, max_pages=max_train_pages
            )
            self.val_dataset = _ASStageBDataset(data_cfg["val_annot_dir"], VAL_CHAPTERS)
            logger.info(
                f"DocDataModule[as]: {len(self.train_dataset)} train / "
                f"{len(self.val_dataset)} val pages"
            )
        else:
            raise ValueError(f"DocDataModule: unknown data_stage={self.data_stage!r}")

    def train_dataloader(self) -> DataLoader:
        if self.train_dataset is None:
            raise RuntimeError("DocDataModule.train_dataloader called before setup()")
        stage_cfg = self.cfg["stage_a"] if self.data_stage == "nist" else self.cfg["stage_b"]
        return DataLoader(
            self.train_dataset,
            batch_size=int(stage_cfg["batch_size"]),
            shuffle=True,
            num_workers=int(stage_cfg.get("num_workers", 0)),
            collate_fn=self.collate_fn,
        )

    def val_dataloader(self) -> DataLoader | None:
        if self.val_dataset is None:
            return None
        stage_cfg = self.cfg["stage_b"]
        return DataLoader(
            self.val_dataset,
            batch_size=int(stage_cfg["batch_size"]),
            shuffle=False,
            num_workers=int(stage_cfg.get("num_workers", 0)),
            collate_fn=self.collate_fn,
        )


In [ ]:
%%writefile src/doc_agent/training/train.py
"""Training — unified entrypoint.

Orchestrates plan.md Step 28's "two-stage fine-tune" as two separate `Trainer.fit()` calls
against the SAME `LitComponent` instance (see lit_modules.py's class docstring for why one
instance, not two): Stage A (all of Step 25's NIST pairs, degraded on-the-fly, no early
stopping — volume/warmup) runs first, then Stage B (the 122 A&S train pages) continues
from Stage A's weights with a lower LR and early-stops on the 20 A&S val pages.
"""

from __future__ import annotations

from pathlib import Path
from typing import Any

import lightning as L

from ..contracts import *  # noqa
from ..logging_conf import get_logger
from .datamodule import DocDataModule, make_collate_fn
from .lit_modules import LitComponent

logger = get_logger(__name__)


def _build_logger(cfg: dict[str, Any], run_name: str) -> Any:
    from lightning.pytorch.loggers import WandbLogger

    log_cfg = cfg.get("logging", {})
    # offline by default: a smoke/CI run must not require WANDB_API_KEY or network access
    # to pass. Step 28's real Kaggle run overrides wandb_mode to "online" once a key is
    # configured in that environment's secrets.
    return WandbLogger(
        project=log_cfg.get("wandb_project", "mathscholar-ocr-finetune"),
        name=run_name,
        mode=log_cfg.get("wandb_mode", "offline"),
    )


def _build_trainer(
    cfg: dict[str, Any],
    stage_cfg: dict[str, Any],
    *,
    early_stopping: bool,
    run_name: str,
    extra_callbacks: list[Any] | None = None,
) -> L.Trainer:
    from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

    callbacks: list[Any] = list(extra_callbacks) if extra_callbacks else []
    if early_stopping:
        callbacks.append(
            EarlyStopping(
                monitor=stage_cfg.get("early_stopping_monitor", "val_loss"),
                patience=int(stage_cfg.get("early_stopping_patience", 3)),
                mode="min",
                # strict=False: don't crash if val_loss hasn't been logged yet. Under real
                # training (full epochs) validation always runs before on_train_epoch_end,
                # so this never fires. It only matters for a smoke/debug run whose
                # max_steps cuts the first epoch short before any validation pass
                # completes -- found running the real smoke train, not assumed -- where
                # strict=True would otherwise crash a run that would have been fine at
                # real scale, for a reason that has nothing to do with the pipeline itself.
                strict=False,
            )
        )
    ckpt_dir = cfg.get("checkpoint", {}).get("dir")
    if ckpt_dir:
        callbacks.append(
            ModelCheckpoint(dirpath=Path(ckpt_dir) / run_name, save_top_k=1, monitor=None)
        )

    trainer_kwargs: dict[str, Any] = {
        "accelerator": "gpu" if str(cfg.get("device", "cpu")).startswith("cuda") else "cpu",
        "devices": 1,
        "max_epochs": int(stage_cfg.get("max_epochs", 1)),
        "logger": _build_logger(cfg, run_name),
        "callbacks": callbacks,
        "deterministic": True,
        "enable_progress_bar": True,
    }
    # Smoke-test / debug overrides: only applied when present in stage_cfg, so the real
    # Step 28 config (no such keys) is unaffected. Applied BEFORE the no-validation force
    # below, not after, so a future stage_a smoke override can never accidentally re-enable
    # a validation pass Stage A has no dataset for (see that block's own comment).
    for key in ("max_steps", "limit_train_batches", "limit_val_batches"):
        if key in stage_cfg:
            trainer_kwargs[key] = stage_cfg[key]

    if not early_stopping:
        # Stage A has no validation split (DocDataModule.val_dataloader() returns None --
        # plan.md Step 27's DECISION: Stage A is off-distribution volume/warmup, not the
        # model-selection signal). Lightning still runs an automatic pre-training "sanity
        # check" validation pass by default regardless of whether early stopping is
        # requested, which crashes on a None dataloader -- found running the real smoke
        # train, not assumed. Both settings below are required: limit_val_batches alone
        # still leaves the sanity check trying to iterate the None dataloader first. This
        # is intentionally the LAST thing set on trainer_kwargs (see above) so it cannot be
        # silently overridden.
        trainer_kwargs["limit_val_batches"] = 0
        trainer_kwargs["num_sanity_val_steps"] = 0

    return L.Trainer(**trainer_kwargs)


def main(component: str, cfg: dict) -> None:
    """Train one component with a seeded Lightning Trainer + W&B logger.

    `component` selects which sub-system to fine-tune; only "ocr" is implemented (plan.md
    Step 27's scope). Runs Stage A then Stage B in sequence — see module docstring.
    """
    L.seed_everything(int(cfg.get("seed", 42)), workers=True)

    lit = LitComponent(cfg, component=component)
    collate_fn = make_collate_fn(lit.processor, cfg["data"]["max_target_length"])

    logger.info("training.train: Stage A (NIST) starting")
    lit.set_stage(cfg["stage_a"])
    dm_a = DocDataModule(cfg, data_stage="nist", collate_fn=collate_fn)
    trainer_a = _build_trainer(cfg, cfg["stage_a"], early_stopping=False, run_name="stage_a_nist")
    trainer_a.fit(lit, datamodule=dm_a)
    logger.info("training.train: Stage A (NIST) complete")

    logger.info("training.train: Stage B (A&S) starting")
    lit.set_stage(cfg["stage_b"])
    dm_b = DocDataModule(cfg, data_stage="as", collate_fn=collate_fn)
    trainer_b = _build_trainer(cfg, cfg["stage_b"], early_stopping=True, run_name="stage_b_as")
    trainer_b.fit(lit, datamodule=dm_b)
    logger.info("training.train: Stage B (A&S) complete")


In [ ]:
%%writefile src/doc_agent/vision/ocr.py
"""Stage 3 — OCR/HTR (BASELINE = pretrained foundation, fine-tuned)"""

from __future__ import annotations

import json
import math
import re
import time
from pathlib import Path
from typing import Any

from ..contracts import *  # noqa
from ..ingest.loader import _chapter_of
from ..logging_conf import get_logger

logger = get_logger(__name__)

# Where a page's own rendered image lives, by convention (not part of cfg): preprocess.py
# (Step 9) writes the deskewed/denoised/CLAHE'd version to data/interim/<page_id>.png;
# get_data.sh (Step 3) writes the raw render to data/pages/<page_id>.png. Interim is
# preferred when present, so OCR always sees the cleaned scan the pipeline actually produced.
INTERIM_DIR = Path("data/interim")
PAGES_DIR = Path("data/pages")

# Per-page cache + sidecars. One <page_id>.mmd holds the whole page's raw Nougat markdown
# (also what data/validate.py's word-count floor reads from), meta.jsonl holds one row per
# CHUNK (ocr_confidence + bbox, summary.md 3f), failures.json logs degenerate pages honestly.
OCR_DIR = Path("data/ocr")
META_PATH = OCR_DIR / "meta.jsonl"
FAILURES_PATH = OCR_DIR / "failures.json"

# Nougat's decoder position limit is 4096 tokens (facebook/nougat-base config). We cap well
# under that: a baseline (not yet fine-tuned) reader running to the true limit on a dense
# numeric-table page is exactly the repetition-degeneration failure mode _is_degenerate()
# exists to catch, and paying that wall-clock cost on CPU for a page we are going to discard
# anyway is wasted. 1536 tokens comfortably covers a normal prose/formula page.
MAX_NEW_TOKENS = 1536

# Step 18b defect 3: generate() set no repetition_penalty, and the spirals in the DEGEN_*
# comments above are exactly what that omission produces. 1.1 is deliberately mild -- A&S
# legitimately repeats subscripts and table rows (a column of "0", a run of "\frac{1}{2}"),
# and HuggingFace's no_repeat_ngram_size would corrupt those outright; a soft per-token
# penalty instead just makes an already-generated token less attractive next time, which
# discourages runaway spirals without forbidding genuine repetition.
REPETITION_PENALTY = 1.1

# Repetition-degeneration guard (summary.md 3a item 4 / plan.md Step 11 point 9): a stuck
# decoder repeats the same short n-gram forever instead of stopping. Detected as the tail of
# the decoded text decomposing into >=MIN_REPEATS consecutive identical NGRAM-word blocks -- a
# strong, cheap signal that needs no external dependency (the `nougat` package's own stopping
# criterion was ruled out project-wide in summary.md 7a for the same reason: dependency
# conflict with this repo's pinned `transformers`).
DEGEN_NGRAM = 12
DEGEN_MIN_REPEATS = 4

# --- three failure modes the tail-only n-gram check above cannot see (found in Step 16) ---
# Measured on Step 16's first Kaggle smoke run: the tail check flagged 1 of 20 pages, while
# 4+ were actually unusable. Each constant below closes one of the gaps that hid them.
#
# 1. Nougat announces its own failures. When it cannot read a page it emits a literal
#    [MISSING_PAGE_POST] / [MISSING_PAGE_EMPTY] / [MISSING_PAGE_FAIL] marker. We were
#    writing those straight to .mmd and counting them as successes -- printed p.243 (a
#    dense table) produced 239 characters consisting of a truncated table header and
#    [MISSING_PAGE_POST], and was reported as a good page.
MISSING_PAGE_RE = re.compile(r"\[MISSING_PAGE[_A-Z]*\]")
#
# 2. A near-empty transcript is a failure, not a short page. Real A&S content pages run
#    to hundreds of characters; the smoke run produced one page of 4 characters and one
#    of 35. The floor sits well under the shortest genuine page observed (239 chars was
#    itself a failure; the shortest sound page was 265).
MIN_PAGE_CHARS = 120
#
# 3. Degeneration ANYWHERE on the page, not just at the tail. The tail check only inspects
#    the last DEGEN_NGRAM * DEGEN_MIN_REPEATS tokens, so a decoder that spirals mid-page and
#    then ends plausibly slips through. Detected as a short character unit repeated many
#    times in a row, which is what these spirals actually look like:
#      - printed p.255 emitted "\!" x603 inside formula 6.1.3, burning the token budget so
#        only 3 of its 14 numbered formulas ever appeared;
#      - printed p.295 read the ch.7 contents list correctly, then ran "<= " to the end.
#    Two weaker signals were measured and REJECTED on the same 19-page sample:
#      - whole-page token diversity: p.255 scored 0.711 unique (threshold would need to be
#        >0.7 to fire) because each "\!\!\!..." run has a different length and so counts as
#        a *distinct* token -- the signal is structurally blind to this failure;
#      - zlib compression ratio: p.255 = 0.183 vs a clean p.065 = 0.224, a margin too thin
#        to set a threshold on without false positives.
#    The repeated-unit count separates cleanly: sound pages topped out at 6 consecutive
#    repeats, the two degenerate pages hit 39 and 38. 20 sits ~3x above the clean maximum
#    and ~2x below the observed failures.
#
#    Step 18b correction: DEGEN_REPEAT_UNIT_MAX_LEN=4 was itself blind to its own dominant
#    failure. Auditing the 594 "successful" Step 16 pages against the PDF's text layer
#    found 41 MORE spiralling pages hiding inside them (91 total; the old detector caught
#    50, i.e. 55%) -- because "\qquad" is 6 characters and "\begin{array}{c}" is 16, both
#    longer than the unit length that could ever match. Widened to 20. That alone would
#    now flag legitimate LaTeX table syntax too -- "c c c c" and "|c|c|c|" are genuine
#    `\begin{tabular}` column specs, not degeneration, and 34 of the original 75 raw hits
#    were exactly this. TABULAR_UNIT_RE excludes any matched unit built ONLY from column-
#    spec characters (alignment letters, bars, braces, digits, whitespace, and "&", the
#    cell separator -- p.328's flagged unit was a bare "&" from a sparse table row, not a
#    spiral) -- a real spiral is always a backslash macro or math content, never just that.
#
#    The repeat count itself was re-checked against Elias's 11 known-genuine spirals and
#    dropped from 20 to 13, for two independent reasons:
#      - exact-match fragility: real spirals decode with a stray whitespace inserted every
#        ~13-14 copies (e.g. "\qquad\qquad...\qquad \qquad..."), which breaks a strict
#        backreference at 20 copies outright -- p.360 and p.177 were both missed this way,
#        p.360 being the exact gold page this repair exists to fix. Matched against the
#        text with ALL whitespace stripped first (LaTeX macros are whitespace-insensitive;
#        a decoder stuck on a token is stuck regardless of incidental spacing), not the
#        word-tokenized `stripped` used by the tail check below.
#      - p.289's genuine spiral only repeats its unit 13 times total, never reaching 20.
#    13 is the lowest threshold that still catches all 11 known cases. Lowering it further
#    starts catching short units (e.g. "\," x14 = ~1% of an otherwise-good page) that read
#    as coincidental formula spacing rather than a stuck decoder, so a MIN_SPIRAL_SPAN_CHARS
#    floor (naturally scaling with unit length) guards against exactly that.
# Step 28 correction (2026-08-12): widened 20 -> 60 after the fine-tuned reader's real
# Kaggle validation run produced a spiral this threshold still missed. `as_p0334`'s
# lowest-scoring prediction (char-F1 0.067, curve point n=122) repeats the unit
# `-\mu xP_{\tau}^{n}(z) ` -- 22 characters, past the old 20-char cap -- more than a dozen
# times, and was scored as a low-quality "success" instead of counted as a failure because
# the detector's own unit-length window couldn't see it. Found by actually reading the
# generated text, not just the aggregate char-F1 number, the same discipline that found
# the original DEGEN_REPEAT_UNIT_MAX_LEN=4 -> 20 gap at Step 18b. 60 gives real headroom
# above the one measured case rather than being set to exactly fit it.
DEGEN_REPEAT_UNIT_MAX_LEN = 60
DEGEN_MIN_UNIT_REPEATS = 13
# Compiles to (.{1,60}?)\1{12,} : a 1-60 character unit, then 12 more copies = 13 total.
DEGEN_REPEAT_RE = re.compile(
    rf"(.{{1,{DEGEN_REPEAT_UNIT_MAX_LEN}}}?)\1{{{DEGEN_MIN_UNIT_REPEATS - 1},}}",
    re.DOTALL,
)
TABULAR_UNIT_RE = re.compile(r"^[lcr|@{}&\s.0-9]*$")
WS_RE = re.compile(r"\s+")
MIN_SPIRAL_SPAN_CHARS = 60

# Step 21 finding 1 (block-level repetition), implemented at Step 28: a WHOLE block
# (paragraph or display equation, separated from its neighbors by a blank line) repeating
# verbatim later in the same page is a different failure shape from DEGEN_REPEAT_RE above
# -- that regex looks for a short-to-medium unit repeating CONSECUTIVELY, not one block
# reappearing once, much later, with different content in between. Measured on the 20
# validation pages: `as_p0340` emits 3 blocks twice (19% of the page duplicated),
# `as_p0441` emits 2 display equations twice (14%) -- both recorded as successes by the
# unit-regex check alone. Exact-match only (not near-duplicate/fuzzy): both measured cases
# are byte-identical repeats, and exact match is the check least likely to false-positive
# on legitimate content that merely looks similar (e.g. two different rows of a table that
# happen to share most of their text).
MIN_BLOCK_DUP_CHARS = 60
_BLOCK_SPLIT_RE = re.compile(r"\n\s*\n")


def _has_duplicate_block(text: str) -> bool:
    """True if any block (paragraph/equation, split on blank lines) of at least
    `MIN_BLOCK_DUP_CHARS` characters appears more than once, verbatim, in `text`."""
    seen: set[str] = set()
    for block in _BLOCK_SPLIT_RE.split(text):
        block = block.strip()
        if len(block) < MIN_BLOCK_DUP_CHARS:
            continue
        if block in seen:
            return True
        seen.add(block)
    return False

# Step 18b defect 5, found on the full-book run (not the 20-page smoke sample): a region
# crop with a near-zero width or height crashes Nougat's OWN preprocessing, not ours.
# layout.detect() (TATR, a learned model) does not guarantee a sane bbox on every region --
# one page produced a crop of shape (1, 1325, 3). HF's image_processing_nougat.crop_margin()
# calls to_channel_dimension_format() on that array; its "channel dim is ambiguous" heuristic
# reads a leading size-1 axis as channels-first, and the resulting transpose((2,0,1)) raises
# `ValueError: axes don't match array` -- an uncaught exception that took the entire ~5h
# Kaggle run down with it (papermill has no per-cell recovery). There is no content to read
# in a 1-pixel-tall sliver anyway, so skip the model call rather than let it reach the crash.
MIN_CROP_DIM_PX = 8

# The citation anchor our Explainable NFR needs (summary.md 3f / 10): A&S formula numbers
# look like "6.1.8". Parsed out of a chunk's OWN text, never guessed.
FORMULA_ID_RE = re.compile(r"\d+\.\d+\.\d+")

# Pinned commit for cfg["ocr"]["model"]'s locked default (facebook/nougat-base) -- bandit
# B615 flags from_pretrained() without a revision as a supply-chain risk, since an unpinned
# model name can resolve to different weights later. Resolved from that repo's `main` ref
# at implementation time; bump deliberately, not implicitly, if it ever needs to move.
NOUGAT_REVISION = "abfecedbb34367c820e233f710fdc7f54e6ab249"


class Reader:
    """Model set by cfg['ocr']. Baseline: pretrained TrOCR/Donut/Tesseract."""

    def __init__(self, cfg: dict) -> None:
        self.cfg = cfg["ocr"]
        self.device = str(cfg.get("device", "cpu"))
        self._model: Any = None
        self._processor: Any = None
        self._dtype: Any = None  # resolved in _ensure_loaded (fp16 on GPU, fp32 on CPU)

    def _ensure_loaded(self) -> None:
        """Load facebook/nougat-base (or cfg['ocr']['model']) on first use, not at
        construction -- so building a Reader() in a test doesn't force a model download."""
        if self._model is not None:
            return
        import torch
        from transformers import NougatProcessor, VisionEncoderDecoderModel

        model_name = self.cfg.get("model", "facebook/nougat-base")
        device = self.device
        if device.startswith("cuda") and not torch.cuda.is_available():
            logger.warning("vision.ocr: cfg requests cuda but no GPU is visible; running on CPU")
            device = "cpu"
        self.device = device

        # Pinned commit for the locked default (bandit B615: an unpinned model name can
        # resolve to different weights later -- same fix vision/layout.py already applies
        # to its own from_pretrained() call, for the same reason). A differently configured
        # model name (not something this project's config.yaml allows) falls back to
        # unpinned, matching from_pretrained's own default resolution.
        revision = NOUGAT_REVISION if model_name == "facebook/nougat-base" else None

        # Half precision on GPU (Step 16). Nougat's own reference implementation runs
        # fp16, and autoregressive decoding is the dominant cost of a full-book pass:
        # Step 16's first Kaggle run measured 17.4 s/page in fp32 on a T4, i.e. ~5.5 h
        # for the 1040-page corpus. CPU stays fp32 -- half precision there is slower,
        # not faster, and unsupported for some ops.
        dtype = torch.float16 if device.startswith("cuda") else torch.float32
        self._dtype = dtype

        self._processor = NougatProcessor.from_pretrained(model_name, revision=revision)
        model = VisionEncoderDecoderModel.from_pretrained(
            model_name, revision=revision, torch_dtype=dtype
        )
        model.eval()
        model.to(device)
        self._model = model

    def _generate(self, image: Any) -> tuple[str, float]:
        """Run one Nougat forward pass on a single image (a full page or a crop).

        Returns (decoded_markdown, confidence). Confidence is the mean per-token
        generation probability (exp of the mean transition log-prob) -- a cheap,
        standard `generate(..., output_scores=True)` readout, not a calibrated metric
        (calibration is the A3 "Calibrated" NFR's job, not this baseline reader's).
        """
        import torch

        self._ensure_loaded()
        # Pixel values must match the model's dtype -- fp16 weights with fp32 inputs
        # raises rather than silently upcasting.
        pixel_values = self._processor(image, return_tensors="pt").pixel_values.to(
            self.device, dtype=self._dtype
        )
        with torch.no_grad():
            outputs = self._model.generate(
                pixel_values,
                min_length=1,
                max_new_tokens=MAX_NEW_TOKENS,
                bad_words_ids=[[self._processor.tokenizer.unk_token_id]],
                repetition_penalty=REPETITION_PENALTY,
                output_scores=True,
                return_dict_in_generate=True,
            )
        sequence = self._processor.batch_decode(outputs.sequences, skip_special_tokens=True)[0]
        sequence = self._processor.post_process_generation(sequence, fix_markdown=False)

        confidence = 0.5  # neutral fallback if the score readout is unavailable
        try:
            # Score the chosen tokens directly instead of calling
            # model.compute_transition_scores(..., normalize_logits=True). That helper
            # reshapes by `self.config.vocab_size`, which a VisionEncoderDecoderConfig
            # does not define -- the decoder's vocabulary lives at
            # config.decoder.vocab_size (50000 for nougat-base). It therefore raised
            # AttributeError on EVERY page and the bare except left confidence pinned at
            # the 0.5 fallback: Step 16's first Kaggle run wrote 201 chunk rows whose
            # ocr_conf was identically 0.5, a constant masquerading as a measurement.
            # Doing the log-softmax ourselves is both correct and version-proof.
            if outputs.scores:
                step_logits = torch.stack(outputs.scores, dim=1)[0].float()  # (steps, vocab)
                gen_ids = outputs.sequences[0, -step_logits.shape[0] :]
                logprobs = torch.log_softmax(step_logits, dim=-1)
                chosen = logprobs[torch.arange(gen_ids.shape[0], device=logprobs.device), gen_ids]
                finite = chosen[torch.isfinite(chosen)]
                if finite.numel() > 0:
                    confidence = float(math.exp(float(finite.mean())))
        except Exception as exc:
            # Log the actual exception. The previous version swallowed it, which is why
            # a per-page failure went unnoticed for an entire GPU run.
            logger.warning(
                f"vision.ocr: confidence unavailable for this page "
                f"({type(exc).__name__}: {exc})"
            )
        return sequence, confidence

    def _generate_region(self, region: Region) -> tuple[str, float]:
        """Crop -> processor -> model.generate -> (decoded text, confidence).

        The shared implementation behind `transcribe_region` (below) and Step 18b defect
        1's page-level retry in `transcribe()`, which needs the confidence value that
        `transcribe_region`'s locked `-> str` signature has nowhere to return.
        """
        from PIL import Image as PILImage

        path = _page_image_path(region.page_id)
        image = PILImage.open(path).convert("RGB").crop(region.bbox)
        if image.width < MIN_CROP_DIM_PX or image.height < MIN_CROP_DIM_PX:
            logger.warning(
                f"vision.ocr: {region.page_id} region bbox={region.bbox} crops to "
                f"{image.width}x{image.height}px (degenerate); skipping the model call"
            )
            return "", 0.0
        return self._generate(image)

    def transcribe_region(self, region: Region) -> str:
        """Crop -> processor -> model.generate -> decoded LaTeX/markdown string.

        Used for (a) per-region re-OCR when a page fails at the page level, and (b) the
        formula-crop (image, latex) pairs the Sprint-4 fine-tune trains on (plan.md 4b) --
        so this stays real and load-bearing, not a shim kept only to satisfy the locked
        `Reader.transcribe_region` signature.
        """
        text, _confidence = self._generate_region(region)
        return text


def _page_image_path(page_id: str) -> Path:
    interim = INTERIM_DIR / f"{page_id}.png"
    if interim.exists():
        return interim
    raw = PAGES_DIR / f"{page_id}.png"
    if raw.exists():
        return raw
    raise FileNotFoundError(
        f"vision.ocr: no image for page_id={page_id!r} under {INTERIM_DIR} or {PAGES_DIR}"
    )


def _group_by_page(regions: list[Region]) -> dict[str, list[Region]]:
    """Group regions by page, preserving first-seen page order and each page's own
    region order (both already reading-order, per vision/layout.py)."""
    groups: dict[str, list[Region]] = {}
    for r in regions:
        groups.setdefault(r.page_id, []).append(r)
    return groups


def _failure_reason(text: str) -> str | None:
    """Why this page's transcript is unusable, or None if it looks sound.

    Returns a short machine-readable reason so data/ocr/failures.json records *how* a
    page failed, not merely that it did -- form Section 5 asks us to report failures
    honestly, and "20% of pages failed, here is the breakdown by mode" is a far more
    useful admission than a bare count. Ordered cheapest check first.
    """
    if MISSING_PAGE_RE.search(text):
        return "nougat-missing-page-marker"

    stripped = text.strip()
    if len(stripped) < MIN_PAGE_CHARS:
        return "empty-or-near-empty"

    # Whole-page repetition (catches mid-page spirals the tail check misses). Matched
    # against the whitespace-collapsed text (see DEGEN_MIN_UNIT_REPEATS above) so a stray
    # space every ~13-14 copies can't break the backreference. Scans every match, not just
    # the first: a page can open with a legitimate tabular block and still spiral later, so
    # stopping at the first hit would let that page through. A MIN_SPIRAL_SPAN_CHARS floor
    # keeps short-unit coincidental repeats (formula spacing like "\," or "\!") from firing
    # on a handful of copies that only cover a sliver of an otherwise-good page.
    no_ws = WS_RE.sub("", stripped)
    for m in DEGEN_REPEAT_RE.finditer(no_ws):
        span = m.end() - m.start()
        if span >= MIN_SPIRAL_SPAN_CHARS and not TABULAR_UNIT_RE.match(m.group(1)):
            return "repetition-degeneration"

    # Block-level repetition (Step 21 finding 1, implemented at Step 28): a whole
    # paragraph/equation block repeating once, verbatim, much later in the page -- a
    # different shape from the consecutive-unit spiral above, so it needs its own check
    # rather than a bigger DEGEN_REPEAT_UNIT_MAX_LEN. See `_has_duplicate_block`'s
    # docstring for the two real pages (as_p0340, as_p0441) that motivated this.
    if _has_duplicate_block(stripped):
        return "block-repetition-degeneration"

    tokens = stripped.split()

    # Original tail check: a decoder still looping when generation was cut off.
    window = DEGEN_NGRAM * DEGEN_MIN_REPEATS
    if len(tokens) >= window:
        tail = tokens[-window:]
        pattern = tail[:DEGEN_NGRAM]
        if all(
            tail[i * DEGEN_NGRAM : (i + 1) * DEGEN_NGRAM] == pattern
            for i in range(1, DEGEN_MIN_REPEATS)
        ):
            return "repetition-degeneration"

    return None


def _is_degenerate(text: str) -> bool:
    """True if this page's transcript is unusable for any reason (see _failure_reason)."""
    return _failure_reason(text) is not None


def _split_markdown_to_regions(markdown: str, n_regions: int) -> list[str]:
    """Approximate a page-level Nougat transcript back onto per-region text.

    Nougat is a PAGE-level model (plan.md Step 11 design note) -- it has no notion of our
    layout regions, so there is no exact mapping. We split the markdown on blank lines
    (Nougat already delimits paragraphs/headings/display-equations that way) and pair the
    resulting blocks with this page's regions **in order** -- both sequences are reading
    order, so position-matching is the best available proxy. Extra trailing blocks are
    folded into the last region rather than dropped; a shortfall pads with "" rather than
    raising, so a page is never lost to a block-count mismatch. This heuristic -- and why
    it's an approximation, not an alignment -- is written up in form Section 3/7.
    """
    blocks = [b.strip() for b in re.split(r"\n\s*\n", markdown.strip()) if b.strip()]
    if n_regions <= 0:
        return []
    if not blocks:
        return [""] * n_regions
    if len(blocks) == n_regions:
        return blocks
    if len(blocks) > n_regions:
        head = blocks[: n_regions - 1]
        tail = "\n\n".join(blocks[n_regions - 1 :])
        return head + [tail]
    return blocks + [""] * (n_regions - len(blocks))


def _chunk_id(doc_id: str, page_id: str, region_idx: int, text: str) -> str:
    base = f"{doc_id}|{page_id}|r{region_idx:02d}"
    m = FORMULA_ID_RE.search(text)
    return f"{base}|{m.group(0)}" if m else base


def _load_jsonl(path: Path) -> dict[str, dict]:
    if not path.exists():
        return {}
    out: dict[str, dict] = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        out[row["chunk_id"]] = row
    return out


def _write_jsonl(path: Path, rows: dict[str, dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows.values():
            f.write(json.dumps(row) + "\n")


def _load_failures(path: Path) -> dict[str, dict]:
    if not path.exists():
        return {}
    try:
        return {row["page_id"]: row for row in json.loads(path.read_text(encoding="utf-8"))}
    except (OSError, json.JSONDecodeError):
        return {}


def _write_failures(path: Path, rows: dict[str, dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(list(rows.values()), indent=2) + "\n", encoding="utf-8")


def _retry_page_by_region(
    reader: Reader, page_regions: list[Region]
) -> tuple[str, list[str], list[float]] | None:
    """Step 18b defect 1 fix: when whole-page generation fails, retry region-by-region
    instead of discarding the page untried. `Reader.transcribe_region`'s own docstring
    already says it exists for exactly this ("per-region re-OCR when a page fails") --
    Step 16 never actually called it, so every failed page was thrown away regardless.

    A single-column region crop is much closer to Nougat's training distribution (modern
    single-column arXiv papers) than a two-column 1964 scan, which is precisely the
    layout defect 4 (early stopping) is measured against -- so this is a real second
    chance, not a formality.

    Returns `(recombined_markdown, region_texts, region_confidences)` -- one text and one
    confidence PER REGION, in reading order, so downstream chunk-building can use the real
    per-region confidence instead of a single page-level scalar -- or `None` if the retry
    is *also* unusable, in which case the caller keeps the original failure.
    """
    region_texts: list[str] = []
    region_confs: list[float] = []
    for region in page_regions:
        # This loop is defect 1's own fix, exercised for the first time at full-book scale
        # in Step 18b -- and it found a crash (defect 5: a degenerate crop dimension, guarded
        # in _generate_region above) that took an entire ~5h unattended run down with it. The
        # per-region guard fixes the KNOWN cause; this except is the belt-and-suspenders for
        # an unknown one -- one bad region among ~1040 pages' worth must not cost the whole
        # job again. Treated the same as a genuinely blank region: empty text, zero confidence.
        try:
            text, conf = reader._generate_region(region)
        except Exception as exc:
            logger.warning(
                f"vision.ocr: region retry crashed on {region.page_id} bbox={region.bbox} "
                f"({type(exc).__name__}: {exc}); treating as empty"
            )
            text, conf = "", 0.0
        region_texts.append(text)
        region_confs.append(conf)
    recombined = "\n\n".join(t for t in region_texts if t.strip())
    if _failure_reason(recombined) is not None:
        return None
    return recombined, region_texts, region_confs


def transcribe(regions: list[Region], cfg: dict, *, limit_pages: int | None = None) -> list[Chunk]:
    """Regions -> text chunks.

    Groups regions by page and runs Nougat **once per page** (not once per region): Nougat
    was trained on whole pages and uses full-page context, so this is both the efficient and
    the accuracy-preserving path (plan.md Step 11 design note). The page's markdown is then
    approximated back onto that page's regions via _split_markdown_to_regions().

    Resumable: a page whose data/ocr/<page_id>.mmd already exists is not re-run through the
    model -- its cached markdown is re-split against the CURRENT regions instead, so a
    layout.py change still produces up-to-date chunks without paying for inference again
    (plan.md Step 11 point 8; matters because Kaggle sessions die at ~9h, summary.md 11.4).

    `limit_pages` is the "test on N pages without running the whole book" guard (plan.md
    Step 11 point 7): an optional keyword-only cap on how many *distinct pages* worth of
    regions are processed, applied AFTER grouping so a partial page is never split across
    the boundary. It defaults to None (no limit) and is not passed by pipeline.py's fixed
    `ocr.transcribe(regions, cfg)` call, so normal pipeline behaviour is unchanged.

    A page whose whole-page decode fails (_failure_reason) is not discarded immediately --
    Step 18b defect 1: it gets ONE region-by-region retry (_retry_page_by_region) before
    being logged to data/ocr/failures.json and producing NO chunks. Only a page that fails
    *both* the whole-page attempt and the region retry is actually given up on -- summary.md
    4i: "mark the page as failed rather than writing garbage" still holds, it just now
    happens after a real second chance, not on the first bad decode.
    """
    reader = Reader(cfg)
    by_page = _group_by_page(regions)
    page_ids = list(by_page)[:limit_pages] if limit_pages is not None else list(by_page)

    OCR_DIR.mkdir(parents=True, exist_ok=True)
    meta_rows = _load_jsonl(META_PATH)
    failure_rows = _load_failures(FAILURES_PATH)

    chunks: list[Chunk] = []
    n_processed = n_cached = n_failed = n_recovered = 0
    t0 = time.time()

    for page_id in page_ids:
        page_regions = by_page[page_id]
        doc_id = _chapter_of(int(page_id[4:])) if page_id.startswith("as_p") else page_id
        mmd_path = OCR_DIR / f"{page_id}.mmd"

        if mmd_path.exists():
            markdown = mmd_path.read_text(encoding="utf-8")
            # Confidence isn't in the .mmd cache (only the text is); recover it from this
            # page's own prior meta.jsonl rows if a previous run already wrote them.
            prior_confs = [
                row["ocr_conf"]
                for cid, row in meta_rows.items()
                if cid.startswith(f"{doc_id}|{page_id}|")
            ]
            confidence = float(prior_confs[0]) if prior_confs else 0.5
            region_texts = _split_markdown_to_regions(markdown, len(page_regions))
            region_confs = [confidence] * len(page_regions)
            n_cached += 1
        else:
            image_path = _page_image_path(page_id)
            from PIL import Image as PILImage

            image = PILImage.open(image_path).convert("RGB")
            markdown, confidence = reader._generate(image)

            reason = _failure_reason(markdown)
            if reason is not None:
                # Defect 1 fix: don't discard the page untried -- retry region-by-region
                # before giving up. A single-column crop is closer to Nougat's training
                # distribution than the two-column 1964 scan that just failed whole.
                retry = _retry_page_by_region(reader, page_regions)
                if retry is None:
                    failure_rows[page_id] = {
                        "page_id": page_id,
                        "reason": reason,
                        "chars": len(markdown.strip()),
                        "detected_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                    }
                    _write_failures(FAILURES_PATH, failure_rows)
                    n_failed += 1
                    logger.warning(
                        f"vision.ocr: {page_id} unusable ({reason}); region retry also failed; skipped"
                    )
                    continue
                markdown, region_texts, region_confs = retry
                n_recovered += 1
                logger.info(
                    f"vision.ocr: {page_id} recovered via region-level retry "
                    f"(page-level attempt: {reason})"
                )
            else:
                region_texts = _split_markdown_to_regions(markdown, len(page_regions))
                region_confs = [confidence] * len(page_regions)

            mmd_path.write_text(markdown, encoding="utf-8")
            n_processed += 1

        for idx, (region, text, conf) in enumerate(
            zip(page_regions, region_texts, region_confs, strict=True)
        ):
            chunk_id = _chunk_id(doc_id, page_id, idx, text)
            chunks.append(
                Chunk(id=chunk_id, doc_id=doc_id, text=text, page_ids=[page_id], score=0.0)
            )
            meta_rows[chunk_id] = {
                "chunk_id": chunk_id,
                "ocr_conf": round(conf, 4),
                "bbox": list(region.bbox),
            }

        if (n_processed + n_cached) % 50 == 0 and n_processed:
            rate = n_processed / max(time.time() - t0, 1e-9)
            logger.info(f"vision.ocr: {n_processed} pages transcribed ({rate:.2f}/s)")

    _write_jsonl(META_PATH, meta_rows)
    logger.info(
        f"vision.ocr: {len(chunks)} chunks from {n_processed} newly-transcribed "
        f"({n_recovered} via region-level retry) + {n_cached} cached pages "
        f"({n_failed} failed/degenerate, skipped)"
    )
    return chunks


In [ ]:
%%writefile configs/train_ocr.yaml
# Step 27 — OCR reader fine-tune config. Read by src/doc_agent/training/{datamodule,
# lit_modules,train}.py. Kept separate from configs/config.yaml for the same reason as
# configs/nist_extract.yaml / configs/degradation.yaml: this is training-run configuration,
# not a knob the retrieval/agent pipeline reads at serve time.
#
# Two-stage curriculum (plan.md Step 28): Stage A (all 695 degraded NIST pairs, no
# validation split -- see below) trains first for volume/warmup, then Stage B (the 122 A&S
# train pages) continues from Stage A's weights with a lower LR and early-stops on the 20
# A&S val pages. Real values here are for the actual Step 28 Kaggle GPU run; Step 27's own
# job is proving this whole pipeline runs end-to-end without crashing (scripts/smoke_train.py
# overrides a handful of these for speed -- see that script, not this file, for the smoke
# numbers).

seed: 42
device: cpu   # Step 28 overrides to "cuda" on Kaggle; local Step 27 dev/smoke stays CPU

ocr:
  model: "facebook/nougat-base"
  # Pinned commit, same reason and same value as vision/ocr.py's NOUGAT_REVISION (bandit
  # B615: an unpinned model name can resolve to different weights later).
  revision: "abfecedbb34367c820e233f710fdc7f54e6ab249"

lora:
  r: 8
  alpha: 16
  dropout: 0.05
  # Optional override of adapt.DEFAULT_LORA_TARGET_KEYWORDS; left unset here so the
  # discovered-from-the-real-model default applies (see adapt.py's own comment for why
  # that default covers both the Swin encoder's and the BART decoder's naming schemes).

data:
  nist_pairs_path: "data/annot/nist/pairs.jsonl"     # Step 25's output -- read-only
  degradation_cfg: "configs/degradation.yaml"        # Step 26's config -- read-only
  train_annot_dir: "data/annot/train"                # Step 22-24's output -- read-only
  val_annot_dir: "data/annot/val"                    # Step 21's output -- read-only
  max_target_length: 1536   # tokens; matches vision/ocr.py's MAX_NEW_TOKENS (same model,
                             # same practical ceiling for a normal prose/formula page)
  # Step 28's learning curve: set to 25 / 50 / 105 / 122 per run (unset/null = all 122).
  # Sorted-prefix subsetting (_ASStageBDataset), so curve points are NESTED (25 ⊂ 50 ⊂
  # 105 ⊂ 122) rather than independently resampled -- never applied to val, which stays
  # the same 20 pages at every curve point.
  stage_b_max_train_pages: null

# --- Stage A: NIST pretraining (volume, no validation split) --------------------------
# Deliberately no early stopping / val here: Stage A is off-distribution (clean pdfTeX
# renders degraded to look like a scan, not a real scan) and exists to teach general
# glyph/layout recognition cheaply, not to be the model selection signal -- that is
# Stage B's job, on real A&S pages. See plan.md Step 27's DECISION for why Stage A's
# per-formula crops don't get a held-out split of their own.
#
# max_epochs raised 1 -> 4 at Step 28 (2026-08-11), together with a real bug fix in
# training/datamodule.py (SeedByEpochCallback): the on-the-fly degradation RNG used to be
# seeded by (seed, pair index) only, identical every epoch regardless of how many ran.
# Fixed so each epoch degrades the same 695 pairs differently -- without that fix, raising
# max_epochs here would have just repeated identical images, not added real exposure. No
# validation split still means this number isn't tuned by a metric; kept modest (not 5+)
# for that reason -- see plan.md Step 28 for the full reasoning.
stage_a:
  batch_size: 4
  max_epochs: 4
  lr: 5.0e-5
  num_workers: 0

# --- Stage B: A&S fine-tune (the target distribution, early-stopped on val) -----------
# max_epochs raised 8 -> 16 at Step 28 (2026-08-11): real evidence from curve point n=25's
# actual Kaggle run showed `Trainer.fit` stopping at `max_epochs=8` with early stopping
# (patience 3) never triggering -- val_loss was still monotonically falling every epoch
# (down to 0.51459 at epoch 7). The old cap was cutting training off before convergence,
# not after; early stopping (unchanged: patience 3 on val_loss) is what actually bounds
# this now, not the epoch count.
stage_b:
  batch_size: 1   # whole-page images; keep small, raise via Kaggle GPU memory at Step 28
  max_epochs: 16
  lr: 2.0e-5
  num_workers: 0
  early_stopping_patience: 3
  early_stopping_monitor: "val_loss"

optimizer:
  weight_decay: 0.01

logging:
  wandb_project: "mathscholar-ocr-finetune"
  # offline by default: a local/CI smoke run must not require a WANDB_API_KEY or network
  # access to pass. Step 28's real Kaggle run sets WANDB_MODE=online (or overrides this
  # key) once a key is configured in that environment's secrets.
  wandb_mode: "offline"

checkpoint:
  dir: "data/models/ocr_lora"   # Step 28 downloads/commits the adapter here (plan.md)


In [ ]:
%%writefile scripts/run_finetune.py
"""Step 28 — the actual Kaggle GPU fine-tune + learning curve.

Step 27 built the pipeline (`doc_agent.training.{datamodule,lit_modules,adapt,train}`) and
proved it runs end-to-end on CPU with `smoke_train.py`. This script RUNS that pipeline at
real scale on a Kaggle GPU: it does not add new training logic, it orchestrates Step 27's
existing pieces the way plan.md Step 28 asks for --- Stage A once, then Stage B four times
(25 / 50 / 105 / 122 A&S train pages), each measured on the same 20 validation pages.

Why this is a separate script and not four calls to `training.train.main()`:
`train.main()` runs Stage A immediately followed by Stage B on ONE `LitComponent`, which is
exactly right for a single run but wrong for a learning curve --- calling it four times would
retrain Stage A four times (four passes over the same 695 NIST pairs, ~4x the GPU time for
zero new information) AND chain each curve point onto the previous one's Stage-B-tuned
weights instead of a clean Stage-A start, which would confound "did 122 help over 105" with
"the model already saw 105 pages of drift before 122 started". Instead:

  1. Stage A trains once. Its LoRA weights are saved with `peft.get_peft_model_state_dict`
     (adapter weights only, ~10-50 MB, not the frozen 350M-param base).
  2. For each curve point, a FRESH `LitComponent` is built (fresh base weights, freshly
     LoRA-wrapped) and Stage A's saved adapter state is loaded into it via
     `peft.set_peft_model_state_dict` before Stage B trains on that curve point's N pages.
     Four independent Stage-B fine-tunes of the same Stage-A start, which is what makes the
     105-vs-122 comparison (plan.md Step 28 point 2) mean what it's supposed to mean.

Resumable across Kaggle's ~9h interactive / ~12h commit ceiling (plan.md Step 28's own
timing note, and plan.md Step 11 point 8's "make the loop resumable" discipline applied
here to training instead of inference): every stage/curve-point boundary is written to
`data/models/ocr_lora/run_state.json` IMMEDIATELY, one point at a time, not batched at the
end --- so a second `kaggle kernels push` after a timeout reads that file first and skips
whatever it already marks done, rather than re-training from zero.

Usage (Kaggle, GPU, from the repo root, `configs/train_ocr.yaml` UNMODIFIED on disk):
    python scripts/run_finetune.py
Measure real per-step GPU time before committing the full curve (plan.md Step 28's own
"measure, don't assume" instruction, same discipline Step 18b should have applied first):
    python scripts/run_finetune.py --measure
Local dry run (CPU, tiny, proves the control flow only --- NOT a real fine-tune):
    python scripts/run_finetune.py --smoke
"""

from __future__ import annotations

import argparse
import json
import math
import os
import sys
import time
from pathlib import Path
from typing import Any

sys.path.insert(0, str(Path(__file__).resolve().parent.parent / "src"))

from doc_agent.data.validate import VAL_CHAPTERS  # noqa: E402
from doc_agent.eval.metrics import exact_formula_match, extract_formulas, ocr_f1  # noqa: E402
from doc_agent.logging_conf import get_logger  # noqa: E402
from doc_agent.training.datamodule import (  # noqa: E402
    DocDataModule,
    SeedByEpochCallback,
    make_collate_fn,
)
from doc_agent.training.lit_modules import LitComponent  # noqa: E402
from doc_agent.training.train import _build_trainer  # noqa: E402
from doc_agent.vision.ocr import (  # noqa: E402
    MAX_NEW_TOKENS,
    REPETITION_PENALTY,
    _failure_reason,
)

logger = get_logger(__name__)

CURVE_POINTS: tuple[int, ...] = (25, 50, 105, 122)
MODELS_DIR = Path("data/models/ocr_lora")
STATE_PATH = MODELS_DIR / "run_state.json"
STAGE_A_CKPT = MODELS_DIR / "stage_a_adapter.pt"
CURVE_FIG_PATH = Path("reports/figures/step28_learning_curve.png")

# Step 18b's own found glyph-confusion class on this typeface, spot-checked here per
# plan.md Step 28's explicit instruction ("grep a handful of validation pages for these
# specific substitutions... before and after each curve point"). A rough presence-count
# proxy, not a token-aligned diff --- consistent with this project's other documented
# heuristics (see vision/ocr.py's _split_markdown_to_regions docstring on why an
# approximation is written up as one, not disguised as an exact measurement).
_NU_RE = "\\nu"
_VEC_RE = "\\vec{"
_ADVANCED_CONSTRUCTS = ("\\sqrt", "\\sum", "\\int", "\\prod")


def _load_yaml(path: str) -> dict[str, Any]:
    import yaml

    with open(path, encoding="utf-8") as fh:
        return yaml.safe_load(fh)


def _atomic_write_json(path: Path, obj: dict[str, Any]) -> None:
    """Write-then-rename so a mid-write interruption (Kaggle session death) can never
    leave `run_state.json` half-written and unreadable by the next resumed push."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2)
        fh.write("\n")
    os.replace(tmp, path)


def _load_state() -> dict[str, Any]:
    if STATE_PATH.exists():
        return json.loads(STATE_PATH.read_text(encoding="utf-8"))
    return {"stage_a_done": False, "curve_points": {}}


def _load_val_records(val_dir: str) -> list[dict[str, Any]]:
    """Gold `{page_id, image_path, text}` rows for the 20 A&S validation pages, read
    directly rather than through `_ASStageBDataset` --- eval needs the raw gold text and
    the un-collated image, neither of which that dataset's `__getitem__` exposes."""
    from doc_agent.ingest.loader import _chapter_of

    records = []
    for jp in sorted(Path(val_dir).glob("*.json")):
        row = json.loads(jp.read_text(encoding="utf-8"))
        png_path = jp.with_suffix(".png")
        if not png_path.exists():
            raise FileNotFoundError(
                f"{jp} has no sibling image {png_path} -- run "
                "`ANNOT=1 bash scripts/get_data.sh` first"
            )
        actual = _chapter_of(row["printed_page"])
        if actual not in VAL_CHAPTERS:
            raise ValueError(f"LEAK — {row['page_id']} (chapter {actual}) is not a VAL chapter")
        records.append({"page_id": row["page_id"], "image_path": str(png_path), "text": row["text"]})
    return records


def _generate_page(lit: LitComponent, image: Any, device: str, max_new_tokens: int) -> str:
    """One Nougat forward pass through the currently-loaded (possibly LoRA-tuned) model.

    Mirrors `vision.ocr.Reader._generate()`'s exact decoding parameters (same
    max_new_tokens/repetition_penalty/bad_words_ids) so a curve point's validation score is
    comparable to the baseline numbers Step 16/18b/21 already measured with that reader ---
    a different decoding config would make "did fine-tuning help" partly a decoding-config
    question instead of a model-quality one.
    """
    import torch

    pixel_values = lit.processor(image, return_tensors="pt").pixel_values.to(device)
    lit.model.eval()
    with torch.no_grad():
        # Generic `PeftModel` (adapt.py's get_peft_model call has no task_type, so this is
        # not a PeftModelForSeq2SeqLM with its own .generate) forwards unknown attributes to
        # the wrapped base model via __getattr__ delegation -- standard, widely-relied-on
        # peft behavior, but not exercised anywhere in Step 27 (training only ever calls
        # forward(), never generate()). Fall back to the explicit path if delegation ever
        # doesn't resolve, rather than crashing the whole curve point on an AttributeError.
        try:
            generate_fn = lit.model.generate
        except AttributeError:
            generate_fn = lit.model.base_model.model.generate
        outputs = generate_fn(
            pixel_values,
            min_length=1,
            max_new_tokens=max_new_tokens,
            bad_words_ids=[[lit.processor.tokenizer.unk_token_id]],
            repetition_penalty=REPETITION_PENALTY,
        )
    text = lit.processor.batch_decode(outputs, skip_special_tokens=True)[0]
    return lit.processor.post_process_generation(text, fix_markdown=False)


def _evaluate_on_val(
    lit: LitComponent, val_records: list[dict[str, Any]], device: str, max_new_tokens: int
) -> dict[str, Any]:
    """Run the just-trained model over all 20 val pages and score it the same way Step 29
    will score the 39 test pages (plan.md's own weighting note): failure rate as its own
    headline number, char-F1 + formula-weighted exact-match among the pages that produced
    output, plus the two Step 18b/28-flagged spot-checks (glyph confusion, advanced-
    construct lag)."""
    from PIL import Image as PILImage

    n = len(val_records)
    n_failed = 0
    f1s: list[float] = []
    exact_num = exact_den = 0.0
    adv_f1s: list[float] = []
    plain_f1s: list[float] = []
    nu_gold_total = nu_pred_v_total = 0
    hallucinated_vec_pages = 0
    per_page: list[dict[str, Any]] = []

    for rec in val_records:
        image = PILImage.open(rec["image_path"]).convert("RGB")
        pred = _generate_page(lit, image, device, max_new_tokens)
        gold = rec["text"]
        reason = _failure_reason(pred)
        # Save the raw prediction, not just its score -- found missing the hard way at
        # Step 28: without it, re-scoring past runs against a corrected `_failure_reason`
        # (as happened here, widened after `as_p0334`'s real spiral slipped through the
        # old 20-char unit cap) requires re-running generate() on every page instead of
        # just re-applying the fixed detector to text already on disk.
        row: dict[str, Any] = {
            "page_id": rec["page_id"], "failed": reason is not None, "pred_text": pred,
        }
        if reason is not None:
            n_failed += 1
            row["failure_reason"] = reason
        else:
            f1 = ocr_f1(pred, gold)
            f1s.append(f1)
            row["char_f1"] = f1
            gold_formulas = extract_formulas(gold)
            weight = len(gold_formulas)
            if weight:
                exact_num += exact_formula_match(pred, gold) * weight
                exact_den += weight
            has_adv = any(c in gold for c in _ADVANCED_CONSTRUCTS)
            (adv_f1s if has_adv else plain_f1s).append(f1)

            nu_gold_total += gold.count(_NU_RE)
            nu_pred_v_total += pred.count(" v ") + pred.count("v)") + pred.count("v(")
            if _VEC_RE in pred and _VEC_RE not in gold:
                hallucinated_vec_pages += 1

        per_page.append(row)
        status = f"FAILED:{reason}" if reason else f"f1={row.get('char_f1', 0):.3f}"
        logger.info(f"run_finetune: eval {rec['page_id']} -> {status}")

    return {
        "n_pages": n,
        "n_failed": n_failed,
        "failure_rate": n_failed / n,
        "mean_char_f1_among_successes": sum(f1s) / len(f1s) if f1s else 0.0,
        "formula_weighted_exact_match": (exact_num / exact_den) if exact_den else 0.0,
        "advanced_construct_mean_f1": sum(adv_f1s) / len(adv_f1s) if adv_f1s else None,
        "plain_mean_f1": sum(plain_f1s) / len(plain_f1s) if plain_f1s else None,
        "glyph_confusion_spotcheck": {
            "nu_occurrences_in_gold": nu_gold_total,
            "bare_v_occurrences_in_pred": nu_pred_v_total,
            "note": "rough presence-count proxy, not a token-aligned diff -- eyeball "
            "per_page below if this looks off",
        },
        "hallucinated_vec_pages": hallucinated_vec_pages,
        "per_page": per_page,
    }


def _save_adapter(lit: LitComponent, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    lit.model.save_pretrained(str(out_dir))  # portable PEFT adapter dir: PeftModel.from_pretrained(base, out_dir)


def _no_ckpt_cfg(cfg: dict[str, Any]) -> dict[str, Any]:
    """A cfg copy with `checkpoint.dir` stripped, so `_build_trainer`'s `ModelCheckpoint`
    callback is never added (its own guard is `if ckpt_dir:` -- see train.py).

    Real Kaggle run, 2026-08-11: `configs/train_ocr.yaml`'s `checkpoint.dir` is
    `data/models/ocr_lora` -- the SAME directory this script's own `_save_adapter` writes
    the small (~10-50 MB) LoRA-only adapter to. Left wired through to `_build_trainer`
    unchanged, Lightning's `ModelCheckpoint` saves a FULL checkpoint there too -- the
    entire 350M-param base model plus optimizer state, multiple GB, once per stage (Stage
    A + 4 curve points = 5x) -- which is what actually filled Kaggle's working disk and
    crashed the run (`OSError: No space left on device`) right after curve point n=25
    finished training, before its eval could even run. We never read that Lightning
    checkpoint (this script's own `run_state.json` + saved adapters ARE the resumable
    state), so it is pure waste here, not a safety net -- disable it entirely rather than
    trying to shrink or rotate it."""
    return {**cfg, "checkpoint": {}}


def _fresh_lit_from_stage_a(cfg: dict[str, Any], stage_a_ckpt: Path) -> LitComponent:
    """A new LitComponent (fresh base weights, freshly LoRA-wrapped) with Stage A's saved
    adapter weights loaded in -- the "clean restart per curve point" described in the
    module docstring."""
    import torch
    from peft import set_peft_model_state_dict

    lit = LitComponent(cfg, component="ocr")
    state_dict = torch.load(stage_a_ckpt, map_location="cpu")
    set_peft_model_state_dict(lit.model, state_dict)
    return lit


def _run_stage_a(cfg: dict[str, Any], collate_fn: Any, state: dict[str, Any]) -> None:
    import torch
    from peft import get_peft_model_state_dict

    if state["stage_a_done"] and STAGE_A_CKPT.exists():
        logger.info("run_finetune: Stage A already done (resumed) -- skipping")
        return

    logger.info("run_finetune: Stage A (NIST, shared pretrain) starting")
    lit = LitComponent(cfg, component="ocr")
    lit.set_stage(cfg["stage_a"])
    dm_a = DocDataModule(cfg, data_stage="nist", collate_fn=collate_fn)
    # Populate dm_a.train_dataset now (Trainer.fit() would call this itself, but
    # SeedByEpochCallback needs a direct reference to the dataset instance to mutate
    # before each epoch -- see that callback's docstring).
    dm_a.setup()
    seed_cb = SeedByEpochCallback(dm_a.train_dataset)
    trainer_a = _build_trainer(
        _no_ckpt_cfg(cfg), cfg["stage_a"], early_stopping=False, run_name="stage_a_nist",
        extra_callbacks=[seed_cb],
    )
    t0 = time.time()
    trainer_a.fit(lit, datamodule=dm_a)
    elapsed = time.time() - t0

    STAGE_A_CKPT.parent.mkdir(parents=True, exist_ok=True)
    torch.save(get_peft_model_state_dict(lit.model), STAGE_A_CKPT)
    state["stage_a_done"] = True
    state["stage_a_elapsed_s"] = elapsed
    _atomic_write_json(STATE_PATH, state)
    logger.info(f"run_finetune: Stage A complete in {elapsed:.1f}s, adapter saved to {STAGE_A_CKPT}")


def _run_curve_point(
    cfg: dict[str, Any], collate_fn: Any, n_pages: int, val_records: list[dict[str, Any]],
    state: dict[str, Any],
) -> None:
    key = str(n_pages)
    if state["curve_points"].get(key, {}).get("done"):
        logger.info(f"run_finetune: curve point n={n_pages} already done (resumed) -- skipping")
        return

    device = cfg.get("device", "cpu")
    logger.info(f"run_finetune: curve point n={n_pages} -- Stage B starting from Stage A weights")
    lit = _fresh_lit_from_stage_a(cfg, STAGE_A_CKPT)
    if device.startswith("cuda"):
        lit = lit.to(device)
    lit.set_stage(cfg["stage_b"])

    run_cfg = dict(cfg)
    run_cfg["data"] = {**cfg["data"], "stage_b_max_train_pages": n_pages}
    dm_b = DocDataModule(run_cfg, data_stage="as", collate_fn=collate_fn)
    trainer_b = _build_trainer(
        _no_ckpt_cfg(cfg), cfg["stage_b"], early_stopping=True, run_name=f"stage_b_n{n_pages}"
    )
    t0 = time.time()
    trainer_b.fit(lit, datamodule=dm_b)
    train_elapsed = time.time() - t0

    # Real Kaggle run, 2026-08-11: Trainer.fit()'s own teardown moved the LightningModule
    # back to CPU after training completed (confirmed by the crash this caused --
    # "Input type torch.cuda.FloatTensor and weight type torch.FloatTensor should be the
    # same" on the very next generate() call, right after an otherwise-clean 8-epoch
    # training run). Lightning does this to free GPU memory once a fit/test/predict call
    # ends; harmless for chained Trainer calls (each re-places the module before running),
    # but this script's eval loop calls generate() directly, outside any Trainer call, so
    # it must re-place the model itself rather than assume fit() left it where training put it.
    if device.startswith("cuda"):
        lit = lit.to(device)

    t0 = time.time()
    metrics = _evaluate_on_val(lit, val_records, device, cfg["data"]["max_target_length"])
    eval_elapsed = time.time() - t0

    adapter_dir = MODELS_DIR / f"curve_n{n_pages}"
    _save_adapter(lit, adapter_dir)

    state["curve_points"][key] = {
        "done": True,
        "n_train_pages": n_pages,
        "train_elapsed_s": train_elapsed,
        "eval_elapsed_s": eval_elapsed,
        "adapter_dir": str(adapter_dir),
        "val_metrics": metrics,
    }
    _atomic_write_json(STATE_PATH, state)
    logger.info(
        f"run_finetune: curve point n={n_pages} done -- failure_rate="
        f"{metrics['failure_rate']:.2f} char_f1={metrics['mean_char_f1_among_successes']:.3f} "
        f"(train {train_elapsed:.0f}s, eval {eval_elapsed:.0f}s)"
    )


def _plot_curve(state: dict[str, Any]) -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    points = sorted(
        (int(k), v["val_metrics"]) for k, v in state["curve_points"].items() if v.get("done")
    )
    if len(points) < len(CURVE_POINTS):
        logger.warning(
            f"run_finetune: only {len(points)}/{len(CURVE_POINTS)} curve points done -- "
            "skipping the plot until the rest finish (re-run this script to continue)"
        )
        return

    ns = [p[0] for p in points]
    f1s = [p[1]["mean_char_f1_among_successes"] for p in points]
    fail_rates = [p[1]["failure_rate"] for p in points]

    fig, ax1 = plt.subplots(figsize=(7, 4.5))
    ax1.plot(ns, f1s, "o-", color="tab:blue", label="mean char-F1 (successes)")
    ax1.set_xlabel("Stage B train pages")
    ax1.set_ylabel("char-F1", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")
    ax1.set_ylim(0, 1)

    ax2 = ax1.twinx()
    ax2.plot(ns, fail_rates, "s--", color="tab:red", label="failure rate")
    ax2.set_ylabel("failure rate (of 20 val pages)", color="tab:red")
    ax2.tick_params(axis="y", labelcolor="tab:red")
    ax2.set_ylim(0, 1)

    fig.suptitle("Step 28 learning curve — validation char-F1 & failure rate vs train-set size")
    fig.tight_layout()
    CURVE_FIG_PATH.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(CURVE_FIG_PATH, dpi=150)
    logger.info(f"run_finetune: learning curve saved to {CURVE_FIG_PATH}")


def _measure(cfg: dict[str, Any], collate_fn: Any) -> None:
    """Real per-step GPU time on a short run, per plan.md Step 28's own instruction not to
    assume a number the way Step 18b's first full-book estimate did. Projects Stage A's
    full-epoch cost + one curve point's worst-case Stage-B cost + eval cost, then the
    likely total for all 4 curve points, so the caller can decide whether to split pushes
    BEFORE committing GPU hours to a run that might not fit Kaggle's ceiling."""
    device = cfg.get("device", "cpu")
    logger.info(f"run_finetune: --measure on device={device}")

    lit = LitComponent(cfg, component="ocr")
    if device.startswith("cuda"):
        lit = lit.to(device)
    lit.set_stage(cfg["stage_a"])
    dm_a = DocDataModule(cfg, data_stage="nist", collate_fn=collate_fn)
    trainer_a = _build_trainer(
        _no_ckpt_cfg(cfg), {**cfg["stage_a"], "max_steps": 5}, early_stopping=False,
        run_name="measure_stage_a",
    )
    t0 = time.time()
    trainer_a.fit(lit, datamodule=dm_a)
    stage_a_step_s = (time.time() - t0) / 5

    run_cfg = dict(cfg)
    run_cfg["data"] = {**cfg["data"], "stage_b_max_train_pages": 25}
    dm_b = DocDataModule(run_cfg, data_stage="as", collate_fn=collate_fn)
    trainer_b = _build_trainer(
        _no_ckpt_cfg(cfg),
        {**cfg["stage_b"], "max_steps": 5, "limit_val_batches": 1},
        early_stopping=False,
        run_name="measure_stage_b",
    )
    t0 = time.time()
    trainer_b.fit(lit, datamodule=dm_b)
    stage_b_step_s = (time.time() - t0) / 5

    # Same Trainer.fit() teardown behavior _run_curve_point hit for real -- re-place
    # before the manual generate() call below, don't assume fit() left it on device.
    if device.startswith("cuda"):
        lit = lit.to(device)

    from PIL import Image as PILImage

    val_records = _load_val_records(cfg["data"]["val_annot_dir"])
    sample_image = PILImage.open(val_records[0]["image_path"]).convert("RGB")
    t0 = time.time()
    _generate_page(lit, sample_image, device, cfg["data"]["max_target_length"])
    generate_s = time.time() - t0

    stage_a_full_steps = math.ceil(695 / int(cfg["stage_a"]["batch_size"]))
    stage_a_total_s = stage_a_full_steps * stage_a_step_s
    per_curve_point_worst_s = (
        max(CURVE_POINTS) * int(cfg["stage_b"]["max_epochs"]) * stage_b_step_s
        + 20 * generate_s
    )
    total_worst_s = stage_a_total_s + len(CURVE_POINTS) * per_curve_point_worst_s

    print("\n=== run_finetune --measure ===")
    print(f"  Stage A: {stage_a_step_s:.2f}s/step measured, {stage_a_full_steps} steps for the "
          f"full 695 pairs -> ~{stage_a_total_s/60:.1f} min")
    print(f"  Stage B: {stage_b_step_s:.2f}s/step measured (batch_size=1)")
    print(f"  eval generate(): {generate_s:.1f}s/page measured (max_new_tokens="
          f"{cfg['data']['max_target_length']})")
    print(f"  worst-case per curve point (max_epochs, no early stop, +20-page eval): "
          f"~{per_curve_point_worst_s/60:.1f} min")
    print(f"  worst-case TOTAL for Stage A + all 4 curve points: ~{total_worst_s/3600:.2f} h")
    print(f"  Kaggle ceiling (plan.md §11.4): ~9h interactive / ~12h commit")
    if total_worst_s > 9 * 3600:
        print("  -> projected total exceeds the interactive ceiling. Plan to run this via "
              "'Save & Run All (Commit)' and/or split curve points across multiple pushes "
              "(this script resumes from data/models/ocr_lora/run_state.json automatically).")
    else:
        print("  -> projected total fits inside a single interactive session, with margin.")


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--cfg", default="configs/train_ocr.yaml")
    p.add_argument("--measure", action="store_true", help="measure real step time, don't train")
    p.add_argument("--smoke", action="store_true", help="tiny CPU dry run of the control flow")
    args = p.parse_args()

    cfg = _load_yaml(args.cfg)

    import torch

    cfg["device"] = "cuda" if torch.cuda.is_available() else "cpu"
    # W&B: online only if a key is actually available (Kaggle Secrets or env) -- a run must
    # not fail for lack of one, same rule train.py's own offline default already follows.
    # "disabled", not "offline", when there's no key: this script never reads the local
    # wandb logs (run_state.json is the real record), and "offline" mode still writes them
    # to disk per stage -- on the real Kaggle run this was one of several contributors to
    # the disk exhaustion that crashed the first push (see _no_ckpt_cfg's docstring for the
    # dominant cause). No local artifact we don't use is worth writing.
    cfg["logging"] = {
        **cfg.get("logging", {}),
        "wandb_mode": "online" if os.environ.get("WANDB_API_KEY") else "disabled",
    }

    if args.smoke:
        cfg["stage_a"] = {**cfg["stage_a"], "batch_size": 1, "max_steps": 3, "max_epochs": 1}
        cfg["stage_b"] = {
            **cfg["stage_b"], "batch_size": 1, "max_steps": 3, "max_epochs": 1,
            "limit_val_batches": 1,
        }
        cfg["data"] = {**cfg["data"], "max_target_length": 128}
        # Real bug, found auditing v5's downloaded output before committing it (2026-08-12):
        # MODELS_DIR itself was never redirected here, only STATE_PATH/STAGE_A_CKPT/
        # CURVE_FIG_PATH -- but `_run_curve_point`'s adapter_dir is built from MODELS_DIR,
        # so every --smoke self-check was writing real (tiny, junk) curve_n5/curve_n8
        # adapter directories straight into the committed data/models/ocr_lora/ path.
        # Redirecting MODELS_DIR too is what actually fixes it; the other three were
        # already correct.
        global CURVE_POINTS, MODELS_DIR, STATE_PATH, STAGE_A_CKPT, CURVE_FIG_PATH
        CURVE_POINTS = (5, 8)
        MODELS_DIR = Path("data/interim/smoke_run_finetune")
        STATE_PATH = MODELS_DIR / "run_state.json"
        STAGE_A_CKPT = MODELS_DIR / "stage_a_adapter.pt"
        CURVE_FIG_PATH = Path("data/interim/smoke_run_finetune/curve.png")

    lit0 = LitComponent(cfg, component="ocr")  # only to build the shared processor for collate_fn
    collate_fn = make_collate_fn(lit0.processor, cfg["data"]["max_target_length"])
    del lit0

    if args.measure:
        _measure(cfg, collate_fn)
        return

    logger.info(f"run_finetune: starting on device={cfg['device']}, "
                f"wandb_mode={cfg['logging']['wandb_mode']}, curve={CURVE_POINTS}")
    state = _load_state()
    _run_stage_a(cfg, collate_fn, state)

    val_records = _load_val_records(cfg["data"]["val_annot_dir"])
    if args.smoke:
        val_records = val_records[:2]  # keep the smoke run fast; not a real eval sample
    for n in CURVE_POINTS:
        _run_curve_point(cfg, collate_fn, n, val_records, state)

    _plot_curve(state)
    logger.info("run_finetune: all curve points complete")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/run_finetune_tables.py
"""Stage C — table-focused continuation fine-tune, from the saved curve_n122 adapter.

Requested directly (not part of plan.md's original Step 25-30 sequence): Step 28's learning
curve showed the fine-tuned reader working well on formulas/prose but hallucinating on
dense numeric tables (`as_p0509`, `as_p0351` -- see plan.md Step 28's writeup). This script
continues fine-tuning FROM the best saved checkpoint (`data/models/ocr_lora/curve_n122`),
table-focused, rather than restarting Stage A/B from scratch.

Training data, two sources, both reused rather than newly risked:
- `data/annot/nist_tables/pairs.jsonl` (`scripts/extract_nist_tables.py`) -- 9 real NIST
  table pairs, degraded on-the-fly like Stage A. Deliberately small: most tables in that
  967-page book are complex multi-line "formula-pair" layouts that the extractor correctly
  refuses to guess at (same "skip rather than scramble" rule Step 25 used) rather than risk
  a silently wrong label -- see that script's own docstring for the measured breakdown.
- `AS_TABLE_DENSE_PAGES` below -- 26 of the 122 A&S train pages already identified as
  genuinely table-dense (>=3 numeric-row-like lines), reused verbatim from
  `data/annot/train/`. REAL, human-verified text, zero new extraction risk, whole-page like
  Stage B -- this is the primary volume here, since 9 NIST pairs alone is not "enough" for
  a meaningful continuation.

Catastrophic-forgetting guard -- the literal ask ("make sure our current model does not
degrade"), not just a training-loss number:
1. The starting checkpoint (`curve_n122`) is never overwritten. This script writes to
   `data/models/ocr_lora/table_ft/`, a new directory.
2. LR is well below Stage B's 2e-5 -- this is a small, delicate nudge on an already-
   converged model, not a fresh curriculum stage.
3. Early stopping monitors `val_loss` on the FULL 20 A&S validation pages, not a
   table-only subset -- a regression on non-table content stops training rather than being
   chased past by continued table-only gradient steps.
4. Before/after comparison reuses `run_finetune.py`'s own `_evaluate_on_val` on the SAME 20
   val pages, printed per-page (not just aggregate) so a hidden regression on any single
   page is visible.

Also runs the NON-training approach recommended alongside this continuation (plan.md
Step 28's table-weakness discussion, option 1): region-level generation on the two known
table-hallucination pages (`as_p0509`, `as_p0351`), using `vision.layout.detect()` to crop
just the table region and running `generate()` on that crop instead of the whole page --
an isolated crop is closer to the model's training distribution (Stage A/C's own crops
are formula/table-sized, not full dense two-column pages). Printed side by side with the
whole-page output for a human read, not scored formally (a region crop's natural output
scope differs from the whole-page gold text, so `ocr_f1` against full-page gold isn't a
fair number here) -- this is exploratory diagnostic evidence, not a pipeline change.

Usage (Kaggle, GPU, from the repo root):
    python scripts/run_finetune_tables.py
Local dry run (CPU, tiny, proves the control flow only):
    python scripts/run_finetune_tables.py --smoke
"""

from __future__ import annotations

import argparse
import json
import os
import sys
import time
from pathlib import Path
from typing import Any

sys.path.insert(0, str(Path(__file__).resolve().parent.parent / "src"))
sys.path.insert(0, str(Path(__file__).resolve().parent))

from doc_agent.logging_conf import get_logger  # noqa: E402
from doc_agent.training.datamodule import make_collate_fn  # noqa: E402
from doc_agent.training.degrade import degrade_one  # noqa: E402
from doc_agent.training.lit_modules import LitComponent  # noqa: E402
from doc_agent.training.train import _build_trainer  # noqa: E402

from run_finetune import (  # noqa: E402
    _evaluate_on_val,
    _load_val_records,
    _no_ckpt_cfg,
    _save_adapter,
)

logger = get_logger(__name__)

# The 26 of 122 A&S train pages measured (>=3 numeric-row-like lines in their own
# human-corrected `text`) to be genuinely table-dense, not just a passing "Table" mention
# -- see the module docstring. `as_p0865`/`as_p0864` (13 rows each) and `as_p1017` (19
# rows) are the densest; all 26 are outside the 20-page VAL set (confirmed: `as_p0509` and
# `as_p0351`, the two pages that motivated this whole continuation, are VAL pages, so
# reusing them here would leak the held-out set -- they are correctly absent below).
AS_TABLE_DENSE_PAGES: tuple[str, ...] = (
    "as_p0024", "as_p0040", "as_p0044", "as_p0050", "as_p0051", "as_p0106", "as_p0114",
    "as_p0115", "as_p0127", "as_p0140", "as_p0141", "as_p0143", "as_p0161", "as_p0181",
    "as_p0748", "as_p0749", "as_p0765", "as_p0813", "as_p0864", "as_p0865", "as_p0915",
    "as_p0998", "as_p1000", "as_p1004", "as_p1010", "as_p1017",
)

# A second run tried diluting this 100% table-shaped mix with 20 non-table A&S pages
# (evenly spread across the book) plus a gentler LR (2.5e-6) and fewer epochs (3), on the
# theory that a 100%-table gradient was specializing away from ordinary formula/prose
# content. That theory was WRONG: the diluted run regressed MORE pages, not fewer
# (failure_rate 0.10->0.25 vs the kept run's 0.10->0.10), including two new failures
# (as_p0453, as_p0518) the kept run didn't have, and it recovered as_p0534 LESS well (still
# failed, vs the kept run's fix to f1=0.480). Whatever is driving the per-page regressions,
# it is not simply "too much table content, too high an LR, too many epochs" -- so this
# script keeps the configuration that produced the better (though still imperfect) result
# rather than the "gentler" one. See plan.md's Step 28 writeup / handoff notes for the full
# before/after comparison table.
NIST_TABLES_PAIRS_PATH = Path("data/annot/nist_tables/pairs.jsonl")
STARTING_ADAPTER_DIR = Path("data/models/ocr_lora/curve_n122")
OUT_DIR = Path("data/models/ocr_lora/table_ft")
STATE_PATH = OUT_DIR / "run_state.json"
LR = 5.0e-6  # well below Stage B's 2e-5 -- see module docstring point 2
MAX_EPOCHS = 6
EARLY_STOPPING_PATIENCE = 2  # tighter than Stage B's 3 -- this is a small delicate nudge,
                             # not a fresh curriculum; stop sooner if val_loss backslides


class _NistTableDataset:
    """Mirrors `training.datamodule._NistStageADataset` for the small NIST table set --
    not reused directly from there because it reads a different pairs.jsonl/out_dir and
    this project's own convention (see `extract_nist_pairs.py` / `extract_nist_tables.py`)
    keeps each extraction's own consumer close to its own script rather than growing one
    shared class's branching."""

    def __init__(self, pairs_path: Path, degradation_cfg: dict[str, Any], seed: int) -> None:
        self._pairs = [json.loads(ln) for ln in pairs_path.read_text(encoding="utf-8").splitlines() if ln.strip()]
        if not self._pairs:
            raise ValueError(f"_NistTableDataset: {pairs_path} is empty")
        self._deg_cfg = degradation_cfg
        self._seed = seed

    def __len__(self) -> int:
        return len(self._pairs)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        import numpy as np
        from PIL import Image

        rec = self._pairs[idx]
        src = Image.open(rec["image"]).convert("L")
        arr = np.asarray(src, dtype=np.float32)
        rng = np.random.default_rng(self._seed + idx)
        degraded = degrade_one(arr, self._deg_cfg, rng)
        image = Image.fromarray(degraded).convert("RGB")
        return {"image": image, "text": rec["text"]}


class _ASTableDataset:
    """The 26 table-dense A&S train pages, reused whole-page like Stage B -- real,
    human-corrected text, no degradation applied (these are real scans already, unlike
    the synthetic NIST crops)."""

    def __init__(self, page_ids: tuple[str, ...], train_dir: str) -> None:
        self._records: list[dict[str, Any]] = []
        for pid in page_ids:
            jp = Path(train_dir) / f"{pid}.json"
            row = json.loads(jp.read_text(encoding="utf-8"))
            png_path = jp.with_suffix(".png")
            if not png_path.exists():
                raise FileNotFoundError(
                    f"_ASTableDataset: {jp} has no sibling image {png_path} -- run "
                    "`ANNOT=1 bash scripts/get_data.sh` first"
                )
            self._records.append({"image_path": str(png_path), "text": row["text"]})

    def __len__(self) -> int:
        return len(self._records)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        from PIL import Image

        rec = self._records[idx]
        image = Image.open(rec["image_path"]).convert("RGB")
        return {"image": image, "text": rec["text"]}


class _ConcatDataset:
    """Minimal concat -- avoids importing torch.utils.data.ConcatDataset just for this."""

    def __init__(self, a: Any, b: Any) -> None:
        self._a, self._b = a, b

    def __len__(self) -> int:
        return len(self._a) + len(self._b)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        if idx < len(self._a):
            return self._a[idx]
        return self._b[idx - len(self._a)]


def _load_yaml(path: str) -> dict[str, Any]:
    import yaml

    with open(path, encoding="utf-8") as fh:
        return yaml.safe_load(fh)


def _atomic_write_json(path: Path, obj: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2)
        fh.write("\n")
    os.replace(tmp, path)


def _lit_from_saved_adapter(cfg: dict[str, Any], adapter_dir: Path) -> LitComponent:
    """A LitComponent whose LoRA weights are the REAL saved curve_n122 adapter, not a
    freshly (randomly) initialized one. `LitComponent.__init__` always builds a fresh
    base + randomly-init LoRA wrap via `apply_lora` (Step 27's own construction path);
    this discards that fresh LoRA wrap and re-wraps the SAME underlying frozen base model
    with the real trained adapter loaded from disk (`PeftModel.from_pretrained`, the
    standard load path for a `save_pretrained` directory -- `_save_adapter` in
    run_finetune.py is what wrote it), rather than re-downloading the base model a second
    time. `is_trainable=True` is required -- `from_pretrained` defaults to False (it's
    built for inference-loading), which silently freezes every LoRA param and left
    `configure_optimizers()` with an empty parameter list (`ValueError: optimizer got an
    empty parameter list`) the first time this ran without it."""
    from peft import PeftModel

    lit = LitComponent(cfg, component="ocr")
    base = lit.model.get_base_model()
    lit.model = PeftModel.from_pretrained(base, str(adapter_dir), is_trainable=True)
    return lit


KNOWN_TABLE_HALLUCINATION_PAGES: tuple[str, ...] = ("as_p0509", "as_p0351")


def _region_level_table_diagnostic(
    lit: LitComponent, device: str, val_annot_dir: str, max_new_tokens: int
) -> None:
    """Non-training comparison: whole-page vs. region-cropped generation, on the two
    pages that motivated this whole continuation. See module docstring for why this is
    printed for a human read rather than scored."""
    from PIL import Image as PILImage

    from doc_agent.contracts import Page
    from doc_agent.vision import layout
    from run_finetune import _generate_page

    print("\n=== Non-training diagnostic: whole-page vs. region-crop generation ===")
    for page_id in KNOWN_TABLE_HALLUCINATION_PAGES:
        jp = Path(val_annot_dir) / f"{page_id}.json"
        if not jp.exists():
            print(f"  {page_id}: not found under {val_annot_dir}, skipping")
            continue
        gold = json.loads(jp.read_text(encoding="utf-8"))["text"]
        png_path = jp.with_suffix(".png")
        full_image = PILImage.open(png_path).convert("RGB")

        whole_page_pred = _generate_page(lit, full_image, device, max_new_tokens)

        page = Page(id=page_id, image_path=str(png_path), doc_id="diagnostic")
        try:
            regions = layout.detect([page], {})
        except Exception as exc:  # layout.detect's real dependencies may not be present
            print(f"  {page_id}: layout.detect() failed ({type(exc).__name__}: {exc}), "
                  "skipping region-crop comparison for this page")
            continue
        table_regions = [r for r in regions if r.kind == "table"]
        if not table_regions:
            print(f"  {page_id}: no table region detected, skipping region-crop comparison")
            continue

        print(f"\n--- {page_id} ---")
        print(f"GOLD (first 300 chars): {gold[:300]}")
        print(f"WHOLE-PAGE pred (first 300 chars): {whole_page_pred[:300]}")
        for i, region in enumerate(table_regions):
            crop = full_image.crop(region.bbox)
            if crop.width < 8 or crop.height < 8:
                continue
            region_pred = _generate_page(lit, crop, device, max_new_tokens)
            print(f"REGION[{i}] bbox={region.bbox} pred (first 300 chars): {region_pred[:300]}")


def main() -> None:
    # _build_trainer sets deterministic=True (same as Stage A/B); Lightning turns that into
    # torch.use_deterministic_algorithms(True), which on this session's cuBLAS build refused
    # to run without this workspace config and raised RuntimeError at the first LoRA forward
    # pass. Must be set before the first CUDA matmul, so as early as possible.
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--cfg", default="configs/train_ocr.yaml")
    p.add_argument("--smoke", action="store_true", help="tiny CPU dry run of the control flow")
    args = p.parse_args()

    cfg = _load_yaml(args.cfg)

    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    cfg["device"] = device
    cfg["logging"] = {
        **cfg.get("logging", {}),
        "wandb_mode": "online" if os.environ.get("WANDB_API_KEY") else "disabled",
    }

    max_epochs = MAX_EPOCHS
    nist_pairs_path = NIST_TABLES_PAIRS_PATH
    ast_page_ids = AS_TABLE_DENSE_PAGES
    starting_adapter = STARTING_ADAPTER_DIR
    out_dir = OUT_DIR
    state_path = STATE_PATH
    if args.smoke:
        max_epochs = 1
        ast_page_ids = ast_page_ids[:2]
        out_dir = Path("data/interim/smoke_table_ft")
        state_path = out_dir / "run_state.json"

    if state_path.exists():
        state = json.loads(state_path.read_text(encoding="utf-8"))
        if state.get("done"):
            logger.info("run_finetune_tables: already done (resumed) -- nothing to do")
            return
    else:
        state = {"done": False}

    logger.info(f"run_finetune_tables: loading starting adapter from {starting_adapter}")
    lit = _lit_from_saved_adapter(cfg, starting_adapter)
    if device.startswith("cuda"):
        lit = lit.to(device)

    collate_fn = make_collate_fn(lit.processor, cfg["data"]["max_target_length"])
    val_records = _load_val_records(cfg["data"]["val_annot_dir"])
    if args.smoke:
        val_records = val_records[:2]

    logger.info("run_finetune_tables: evaluating BEFORE continuation (baseline = curve_n122)")
    t0 = time.time()
    before_metrics = _evaluate_on_val(lit, val_records, device, cfg["data"]["max_target_length"])
    logger.info(
        f"run_finetune_tables: BEFORE -- failure_rate={before_metrics['failure_rate']:.2f} "
        f"char_f1={before_metrics['mean_char_f1_among_successes']:.3f} "
        f"({time.time() - t0:.0f}s)"
    )
    state["before_metrics"] = before_metrics
    _atomic_write_json(state_path, state)

    degradation_cfg = _load_yaml(cfg["data"]["degradation_cfg"])
    nist_ds = _NistTableDataset(nist_pairs_path, degradation_cfg, cfg["seed"])
    ast_ds = _ASTableDataset(ast_page_ids, cfg["data"]["train_annot_dir"])
    train_ds = _ConcatDataset(nist_ds, ast_ds)
    logger.info(f"run_finetune_tables: table training set = {len(nist_ds)} NIST + "
                f"{len(ast_ds)} A&S = {len(train_ds)} pairs")

    import lightning as L

    class _TableDataModule(L.LightningDataModule):
        """val comes from the SAME 20 A&S val pages every other stage uses
        (catastrophic-forgetting guard, see module docstring point 3), not a table-only
        subset. Must subclass L.LightningDataModule (not a bare protocol class) --
        Trainer.fit()'s internal is_overridden() hook check requires a real Lightning
        parent class and raises ValueError before any training step otherwise."""

        def __init__(self) -> None:
            super().__init__()
            self.train_dataset = train_ds

        def train_dataloader(self):  # noqa: ANN201
            from torch.utils.data import DataLoader

            return DataLoader(
                train_ds, batch_size=1, shuffle=True, num_workers=0, collate_fn=collate_fn,
            )

        def val_dataloader(self):  # noqa: ANN201
            from torch.utils.data import DataLoader

            class _ValWrap:
                def __len__(self_inner) -> int:
                    return len(val_records)

                def __getitem__(self_inner, idx: int) -> dict[str, Any]:
                    from PIL import Image

                    rec = val_records[idx]
                    image = Image.open(rec["image_path"]).convert("RGB")
                    return {"image": image, "text": rec["text"]}

            return DataLoader(
                _ValWrap(), batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_fn,
            )

    lit.set_stage({"lr": LR})
    stage_cfg = {
        "lr": LR, "max_epochs": max_epochs,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "early_stopping_monitor": "val_loss",
    }
    if args.smoke:
        stage_cfg["max_steps"] = 3
        stage_cfg["limit_val_batches"] = 1
    trainer = _build_trainer(
        _no_ckpt_cfg(cfg), stage_cfg, early_stopping=True, run_name="stage_c_tables",
    )
    t0 = time.time()
    trainer.fit(lit, datamodule=_TableDataModule())
    train_elapsed = time.time() - t0

    if device.startswith("cuda"):
        lit = lit.to(device)  # Trainer.fit's own teardown can move it back to CPU

    logger.info("run_finetune_tables: evaluating AFTER continuation")
    t0 = time.time()
    after_metrics = _evaluate_on_val(lit, val_records, device, cfg["data"]["max_target_length"])
    eval_elapsed = time.time() - t0
    logger.info(
        f"run_finetune_tables: AFTER -- failure_rate={after_metrics['failure_rate']:.2f} "
        f"char_f1={after_metrics['mean_char_f1_among_successes']:.3f} "
        f"(train {train_elapsed:.0f}s, eval {eval_elapsed:.0f}s)"
    )

    _save_adapter(lit, out_dir)

    if not args.smoke:
        _region_level_table_diagnostic(
            lit, device, cfg["data"]["val_annot_dir"], cfg["data"]["max_target_length"]
        )

    # Per-page before/after diff -- the real regression check, not just aggregate deltas.
    before_by_page = {r["page_id"]: r for r in before_metrics["per_page"]}
    after_by_page = {r["page_id"]: r for r in after_metrics["per_page"]}
    regressions = []
    for pid, after_row in after_by_page.items():
        before_row = before_by_page.get(pid, {})
        before_f1 = before_row.get("char_f1", 0.0 if before_row.get("failed") else None)
        after_f1 = after_row.get("char_f1", 0.0 if after_row.get("failed") else None)
        if before_f1 is not None and after_f1 is not None and after_f1 < before_f1 - 0.05:
            regressions.append({"page_id": pid, "before_f1": before_f1, "after_f1": after_f1})

    state.update({
        "done": True,
        "train_pairs": {"nist": len(nist_ds), "as_table_dense": len(ast_ds)},
        "train_elapsed_s": train_elapsed,
        "eval_elapsed_s": eval_elapsed,
        "before_metrics": before_metrics,
        "after_metrics": after_metrics,
        "regressions_over_0.05_f1": regressions,
        "adapter_dir": str(out_dir),
    })
    _atomic_write_json(state_path, state)

    print("\n=== Stage C (table-focused continuation) summary ===")
    print(f"BEFORE: failure_rate={before_metrics['failure_rate']:.2f}  "
          f"char_f1={before_metrics['mean_char_f1_among_successes']:.3f}")
    print(f"AFTER:  failure_rate={after_metrics['failure_rate']:.2f}  "
          f"char_f1={after_metrics['mean_char_f1_among_successes']:.3f}")
    if regressions:
        print(f"\n⚠️  {len(regressions)} page(s) regressed by >0.05 char-F1:")
        for r in regressions:
            print(f"  {r['page_id']}: {r['before_f1']:.3f} -> {r['after_f1']:.3f}")
    else:
        print("\nNo page regressed by more than 0.05 char-F1.")


if __name__ == "__main__":
    main()


## 6. Smoke-check, then the real run

In [ ]:
!python scripts/run_finetune_tables.py --smoke


In [ ]:
!df -h /kaggle/working
!python scripts/run_finetune_tables.py
!df -h /kaggle/working


## 7. Package the output for download

In [ ]:
import shutil
out_dir = "/kaggle/working/out"
os.makedirs(out_dir, exist_ok=True)
shutil.copytree("data/models/ocr_lora/table_ft", f"{out_dir}/table_ft", dirs_exist_ok=True)
archive_path = shutil.make_archive("/kaggle/working/stage_c_output", "zip", out_dir)
print(f"wrote {archive_path} ({os.path.getsize(archive_path) / 1e6:.1f} MB)")
